# 02 · How many dimensions? sEMG synergies and eigenpostures

**Investigation ladder** (see [`EVALUATION.md`](../EVALUATION.md)):
- **Stage 1 · Existence: is the EMG low-dimensional beyond chance?**

**Read first:** [Lee & Seung 1999](https://www.nature.com/articles/44565) (NMF), [Santello et al. 1998](https://www.jneurosci.org/content/18/23/10105) (eigenpostures), Cunningham & Yu 2014, *Dimensionality reduction for large-scale neural recordings* (Nat Neurosci). **Owner:** Person B

**We expect:** 3 to 5 NMF synergies explain the held-out EMG (VAF ≥ 90%), clearly better than random synergies of the same size, and k stays the same when whole stages are held out. **Wrong if:** we need close to 16 synergies, or random non-negative synergies do almost as well.

**Model:** `EMG ≈ C × W`: `W` (k × 16) holds the k synergies, `C` (T × k) says how active each one is at every
frame. You write **NMF and PCA yourself, in NumPy** (no scikit-learn for the core algorithms). NMF comes
first, derived step by step from its objective: loss → gradient → why gradient descent struggles →
multiplicative updates → a test on synergies we planted.

**How this notebook works** (the same in every notebook)
- **`# TODO`** cells are yours. Each has a docstring with the input and output shapes. You write **every
  function and every plot** yourself in NumPy: the algorithms, the test tools (numerical gradients, synthetic
  data, synergy matching), the folds and the confidence intervals. Only the setup cells and the ✅ checks are given.
- Under every TODO there are three **💡 hints** that open one at a time: the plan in words, the tools with a tiny
  example, then a skeleton with blanks (`____`).
- After each TODO a **✅ check cell** compares your answer with the reference in [`ref02.py`](ref02.py), then a **📊 plot**
  (also yours, from a spec) shows what your code did. Other notebooks use the reference, so your blanks never break theirs.
- **✍️ questions** are for your own words, in the markdown cell under them.
- The **Result** cell at the end is the **gate**: it decides whether the next stage of the investigation is worth running.

## 🎯 Goal and milestones
*Draft milestones: agree them as a team on Friday, then edit here.*

**Proposal:** ✅ **Aim 1** (weeks 1–5): define stable, low-dimensional representations of sEMG *and* hand pose.

**Goal:** choose the two bottleneck sizes: **k** synergies for the sEMG input and **K_POSE** eigenpostures for the pose output, each with a fair reference point.

**Why:** H1 says squeezing both sides of the decoder through small representations cuts calibration on new users. Every later stage uses the k chosen here, so it must not depend on a generous metric or on the split.

**Milestones**
0. NMF written from its objective and gradient, and shown to recover planted synergies
1. VAF vs k (k = 1 to 10) for NMF and PCA on test data, uncentred **and** centred
2. The same curve for random non-negative synergies (the null)
3. k chosen by leave-one-stage-out, with a 95% CI over held-out stage groups
4. A figure of the chosen synergies, and pose error (°) vs number of eigenpostures

**Gate (decides the next stage):** NMF beats random synergies by a clear margin (CI of the gap above 0) at the k chosen by leave-one-stage-out. If not, stop calling them synergies (stage 2 and 3 questions change).

**Done when:** a chosen k and K_POSE, each with its curve, its null and its CI.

In [2]:
# ── Setup: works on Colab and locally ─────────────────────────────────────────
# After this cell the working directory is the repo root, wherever you run from.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    ROOT = Path("/content/emg2pose")
    if not ROOT.exists():
        !git clone https://github.com/Wilbertbh-Tan/emg2pose.git {ROOT}
    else:
        !git -C {ROOT} pull -q                 # pick up files pushed since the first clone
    !pip install -q h5py
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    assert (ROOT / "setup.py").exists(), "Run this notebook from inside the emg2pose repo"
os.chdir(ROOT)

_d = ROOT / "notebooks" / "experiments" / "02_how_many_synergies"
assert (_d / "ref02.py").exists(), f"{_d}/ref02.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref02", None)
_d = ROOT / "notebooks" / "experiments" / "common"
sys.path.insert(0, str(_d)); sys.modules.pop("protocol", None)
import protocol                 # shared evaluation protocol: sessions, folds, baselines, nulls, CIs
import ref02 as ref          # the reference answers, used only by the ✅ check cells

PROCESSED = ROOT / "data" / "processed_mini.npz"   # made by notebook 01's prep.py (data/ is never committed)

In [3]:
# ── Load the shared data (builds it with prep.py the first time) ─────────────
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(ROOT / "notebooks" / "experiments" / "01_data_prep")); sys.modules.pop("prep", None)
import prep
if not prep.is_current(PROCESSED):   # missing, or built by an older version of prep.py
    subprocess.run([sys.executable, "notebooks/experiments/01_data_prep/prep.py"], check=True)
d = np.load(PROCESSED)

emg        = d["emg"].astype(np.float64)   # (N, 16)  EMG envelope, 50 samples/s, non-negative
pose       = d["pose"].astype(np.float64)  # (N, 20)  joint angles in degrees
pose_valid = d["pose_valid"]               # (N,)     False where pose tracking failed
rec        = d["rec"]                      # (N,)     which recording (0–29) each row comes from
split      = d["split"]                    # (N,)     0 = train, 1 = test, -1 = gap (use neither)
stage_of_rec, side_of_rec = d["stage"], d["side"]   # (30,) task name and hand per recording
filename_of_rec = d["filename"]                   # (30,) file name: pairs the two simultaneous recordings

train, test = split == 0, split == 1
SEED = 0   # pass seed=SEED to every random start so results repeat
print(f"{len(emg)} frames | {train.sum()} train | {test.sum()} test | {len(stage_of_rec)} recordings")

105484 frames | 84376 train | 19608 test | 30 recordings


In [ ]:
# ── A helper for the ✅ check cells: prints what your code produced, next to what was expected ──
def show(label: str, yours: object, expected: object = None) -> None:
    """Print one of your results (and the expected value, if given), so you can see each check working."""
    def fmt(v: object) -> str:
        if isinstance(v, str):
            return v
        if isinstance(v, np.ndarray):
            if v.size <= 32:
                return f"shape {v.shape}\n" + np.array2string(v, precision=4, suppress_small=True, max_line_width=100)
            return f"shape {v.shape}, first entries {np.array2string(v.ravel()[:6], precision=4)} …"
        if isinstance(v, (float, np.floating)):
            return f"{float(v):.6g}"
        return repr(v)
    pad = "\n" + " " * 14
    print(f"• {label}")
    print("    yours:    " + fmt(yours).replace("\n", pad))
    if expected is not None:
        print("    expected: " + fmt(expected).replace("\n", pad))

### The variables you start with
`N` = number of frames in all 30 recordings together (105,484: about 35 minutes at 50 frames/s).
Every row of every array is one frame, 20 ms long. Boolean arrays are **masks**: `emg[train]` keeps only the train rows.

| Name | Shape | dtype | What it is |
|---|---|---|---|
| `emg` | (N, 16) | float64 | EMG envelope, one column per electrode, ≥ 0, roughly 0–1 |
| `pose` | (N, 20) | float64 | joint angles in **degrees**, one column per joint |
| `pose_valid` | (N,) | bool | False where pose tracking failed (don't use those `pose` rows) |
| `rec` | (N,) | int16 | which recording (0–29) the frame comes from |
| `split` | (N,) | int8 | 0 = train, 1 = test, −1 = gap between them (use neither) |
| `train`, `test` | (N,) | bool | `split == 0`, `split == 1` |
| `stage_of_rec` | (30,) | str | task name of each recording; per frame: `stage_of_rec[rec]` → (N,) |
| `side_of_rec` | (30,) | str | `"left"` or `"right"` per recording; per frame: `side_of_rec[rec]` → (N,) |
| `filename_of_rec` | (30,) | str | file name per recording. The left and right recordings with the same block number were made **at the same time**, so the 30 recordings are only **15 independent blocks** |

Letters used in the docstrings below: **T** = number of frames in the array you pass in (any subset of N),
**D** = number of columns (16 EMG channels or 20 joints), **k** = number of components.
A good habit: try each function on a tiny made-up array first (e.g. `np.random.rand(5, 16)`).

## Before you run anything: your prediction

Write what you expect and why (2–3 sentences). Compare at the end. Surprises are the interesting part.

> *Prediction:* I think the structure should be confined to some latent states that are reflective othe muscles. I think geometric should reflect better and if there is some way to include some time variance that would be work even better. 

---
## Task 1 · NMF from its mathematical foundation
A **muscle synergy** is a fixed pattern of activity across the 16 electrodes that the nervous system switches on
and off as one unit. If the forearm really works that way, every frame of EMG is a **weighted sum of a few
patterns**, with weights that can only be zero or positive:

$$\underbrace{X}_{T \times 16} \;\approx\; \underbrace{C}_{T \times k}\;\underbrace{W}_{k \times 16},
\qquad C \ge 0,\; W \ge 0$$

- Row *i* of `W` is synergy *i*: how strongly each electrode takes part in it.
- Column *i* of `C` is its **activation**: how strongly synergy *i* is switched on at every frame.
- **Why non-negative?** The envelope measures how hard muscles are working. Muscles can add activity, never
  subtract it, so neither a pattern nor its activation should ever be negative. (PCA, in Task 3, has no such rule.)

This is **non-negative matrix factorisation (NMF)**. You will not be handed the algorithm. You will **derive
it** in five steps (plus a sixth, for practice), writing one small function at each step, and each function is used by the next:

| Step | Question | You write |
|---|---|---|
| **1a** | What exactly are we minimising? | `nmf_loss` |
| **1b** | Which way is "downhill"? (and how do we check a gradient?) | `numerical_grad`, `nmf_grads` |
| **1c** | Why doesn't plain gradient descent just work? | `pgd_step`, `safe_steps`, `run_pgd` |
| **1d** | Which step size makes the problem go away? | `update_W`, `update_C` |
| **1e** | The whole algorithm, and does it find synergies we planted? | `nmf_init`, `nmf_fit`, `nmf_transform`, `planted_synergies`, `match_synergies` |
| **1f** | *(practice)* What if the noise isn't Gaussian? KL-divergence NMF | `kl_loss`, `kl_grads`, `kl_update_W`, `kl_update_C`, `nmf_fit_kl` |

**Notation.** Entries are written with indices: $X_{tj}$ is frame *t*, electrode *j*; $(CW)_{tj} =
\sum_i C_{ti} W_{ij}$ is the rebuilt value. `⊙` and fractions of matrices act **entry by entry** (`*` and `/` in
NumPy); written-together matrices are matrix products (`@`).

### Step 1a · The objective: what are we minimising?
**The goal.** Find `C` and `W` (both ≥ 0) so that the rebuild `C W` is as close as possible to the recorded `X`.
A computer can't search for "as close as possible" until "how far is `C W` from `X`?" is **one number**. That
number is the **loss** `L`: smaller means closer, 0 means a perfect rebuild. Everything after this step (the
gradient, the updates, the fit) is a way of making `L` smaller.

**Building the loss, one entry at a time.**
1. **What was recorded:** $X_{tj}$, the EMG at frame *t*, electrode *j*.
2. **What the model rebuilds:** $(CW)_{tj} = \sum_i C_{ti}\,W_{ij}$. For each synergy *i*, take how strongly it
   uses electrode *j* ($W_{ij}$) times how active it is at frame *t* ($C_{ti}$), and add over the synergies.
3. **The miss, called the residual:** $r_{tj} = X_{tj} - (CW)_{tj}$. Positive means the rebuild is too small;
   negative means it is too big.
4. **Square every miss, add them all up, and halve:**

$$L(C, W) \;=\; \tfrac12 \sum_{t,j} r_{tj}^2 \;=\; \tfrac12 \sum_{t,j} \big(X_{tj} - (CW)_{tj}\big)^2$$

The sum runs over **every** entry: all T frames × all 16 electrodes give one number for the whole recording.

**A worked example** (2 frames, 2 electrodes, k = 1 synergy). $X = \begin{bmatrix}2 & 1\\ 4 & 3\end{bmatrix}$,
$C = \begin{bmatrix}1\\ 2\end{bmatrix}$ (frame 2 is twice as active), $W = \begin{bmatrix}2 & 0.5\end{bmatrix}$ (the synergy uses electrode 1 more).

| entry (t, j) | recorded $X_{tj}$ | rebuild $C_{t1} W_{1j}$ | residual $r_{tj}$ | $r_{tj}^2$ |
|---|---|---|---|---|
| (1, 1) | 2 | 1 × 2 = 2 | 0 | 0 |
| (1, 2) | 1 | 1 × 0.5 = 0.5 | 0.5 | 0.25 |
| (2, 1) | 4 | 2 × 2 = 4 | 0 | 0 |
| (2, 2) | 3 | 2 × 0.5 = 1 | 2 | 4 |
| | | | **sum** | **4.25** |

So $L = \tfrac12 \times 4.25 = 2.125$. Notice that the one big miss (2) supplies 4 of the 4.25. Squaring lets big
errors dominate; the next cells explain why that is (mostly) what we want. The check cell tests your function on
exactly this example.

**The same thing written as a norm.** The **Frobenius norm** of a matrix is its length, as if you laid all its entries out in one
long vector and used Pythagoras:
$$\lVert A \rVert_F = \sqrt{\textstyle\sum_{t,j} A_{tj}^2}$$
In the example, $\lVert X - CW \rVert_F = \sqrt{4.25} \approx 2.06$, the "distance" between the data and the rebuild. The loss is
$$L = \tfrac12\,\lVert X - CW \rVert_F^2$$
- **The square** removes the square root. A root never changes which of two values is smaller, so the best
  `C, W` stays the same, and without it the derivative is much simpler.
- **The ½ is not part of the norm.** We add it ourselves, so that the derivative comes out clean (reason 4 below).

**You write** `nmf_loss`: these four steps in one line of NumPy.

In [4]:
def nmf_loss(X: np.ndarray, C: np.ndarray, W: np.ndarray) -> float:
    """The NMF objective L = 1/2 * sum over every entry of (X - C @ W)^2.

    Parameters
    ----------
    X : (T, D) float, the data (non-negative)
    C : (T, k) float, activations
    W : (k, D) float, synergies

    Returns
    -------
    float: half the sum of squared errors. 0 means C @ W rebuilds X exactly.
    """
    return 0.5 * np.sum((X - C @ W) ** 2)


#### 💡 Stuck on step 1a? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Rebuild the data: `C @ W` has the same shape as `X`.
2. Subtract, square every entry, add up **all** the squares into one number.
3. Multiply by ½ and return it as a plain `float`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `C @ W` is a matrix product: (T, k) @ (k, D) → (T, D).
- `np.sum(a ** 2)` with no `axis=` adds every entry into one number.

```python
C = np.array([[1.0], [2.0]]); W = np.array([[1.0, 0.5]])
C @ W                      # [[1. , 0.5], [2. , 1. ]]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def nmf_loss(X, C, W):
    return float(0.5 * np.sum((____ - ____ @ ____) ** 2))
```

</details>

In [ ]:
# ✅ Check step 1a
rng_t = np.random.default_rng(0)
X_t, C_t, W_t = rng_t.random((6, 5)), rng_t.random((6, 2)), rng_t.random((2, 5))   # a tiny problem
L_t = nmf_loss(X_t, C_t, W_t)
show("loss on a tiny random problem (6 frames × 5 electrodes, k = 2)", L_t, ref.nmf_loss(X_t, C_t, W_t))
show("loss when X is exactly C @ W", nmf_loss(C_t @ W_t, C_t, W_t), 0.0)
assert isinstance(L_t, float), f"return a plain float (got {type(L_t).__name__}): wrap the result in float(...)"
assert nmf_loss(C_t @ W_t, C_t, W_t) == 0, "if X is exactly C @ W the loss should be 0"
assert abs(L_t - 2 * ref.nmf_loss(X_t, C_t, W_t)) > 1e-9, "your loss is twice the reference: the 1/2 is missing"
assert abs(L_t - ref.nmf_loss(X_t, C_t, W_t)) < 1e-12, f"loss {L_t:.6f}, expected {ref.nmf_loss(X_t, C_t, W_t):.6f}"

X_ex, C_ex, W_ex = np.array([[2.0, 1.0], [4.0, 3.0]]), np.array([[1.0], [2.0]]), np.array([[2.0, 0.5]])
show("the worked example from the markdown", nmf_loss(X_ex, C_ex, W_ex), 2.125)
assert abs(nmf_loss(X_ex, C_ex, W_ex) - 2.125) < 1e-12, \
    f"the worked example gives L = {nmf_loss(X_ex, C_ex, W_ex)}, expected 2.125 (½ × (0 + 0.25 + 0 + 4))"

X_sub = emg[train][:20000]                       # 20,000 training frames: the data for steps 1c–1e
C0, W0 = ref.nmf_init(X_sub, 4, SEED)            # a random non-negative start with k = 4 (step 1e builds this)
print(f"✅ nmf_loss works. On 20,000 EMG frames: random start L = {nmf_loss(X_sub, C0, W0):,.0f}; "
      f"rebuilding everything as 0 gives L = {nmf_loss(X_sub, 0 * C0, W0):,.0f}")

### Why squared error? Four reasons, in detail

#### 1 · Errors in both directions must count, and big ones count more
**Why not just add up the residuals?** They can be positive or negative, so they cancel. Residuals of +2 and −2
add up to 0, and a bad fit would score as perfect. The loss must turn every miss into a cost, whichever direction
it goes. The two common ways:

| loss per entry | misses (1, 1) | misses (2, 0) | which fit is worse? |
|---|---|---|---|
| absolute, $\lvert r\rvert$ | 1 + 1 = **2** | 2 + 0 = **2** | equally bad |
| squared, $r^2$ | 1 + 1 = **2** | 4 + 0 = **4** | one big miss is worse |

With squares, a miss twice as big costs **four** times as much. So squared error prefers to be a little off
everywhere rather than badly off in one place.

**The price is outliers.** One artefact spike with residual 10 costs 100: as much as 100 entries that are each
off by 1. The fit bends towards it. The simplest case: fit one constant $c$ to the numbers `[1, 2, 3, 10]`.
- Smallest sum of **squares** → $c$ = the **mean**, 4. The 10 drags it up.
- Smallest sum of **absolute** values → $c$ = the **median**, 2.5. The 10 barely matters.

This is one reason notebook 01 cleans and smooths the EMG before anything is fitted.

#### 2 · It is the right loss if the noise is Gaussian (maximum likelihood)
This is the deepest reason. It shows that squared error follows from **an assumption about the noise**.

**(a) What a density says.** A probability density $p(x)$ says how likely each value $x$ is: where the curve is
high, values turn up often. The **Gaussian** (normal, bell-curve) density with centre $\mu$ and spread $\sigma$ is
$$p(x) = \underbrace{\frac{1}{\sqrt{2\pi\sigma^2}}}_{\text{height}}\;
\exp\!\Big(\underbrace{-\frac{(x - \mu)^2}{2\sigma^2}}_{\text{shape}}\Big)$$
Read the shape part from the inside out:
1. $x - \mu$: how far $x$ is from the centre.
2. $(x - \mu)^2$: that distance squared. It is the same for "too high" and "too low".
3. $\div\, 2\sigma^2$: the distance measured against the spread. A distance of 2 is huge if σ = 0.5 and tiny if σ = 10.
4. $\exp(-\dots)$: turns "far" into "unlikely". At the centre, exp(0) = 1 (the peak); 1σ away, exp(−½) ≈ 0.61;
   2σ away, exp(−2) ≈ 0.14; 3σ away, exp(−4.5) ≈ 0.011.

The height factor only scales the curve so its total area is 1; it doesn't depend on $x$.

**(b) Put it on the data.** The model: every recorded entry is the rebuild plus bell-curve noise,
$$X_{tj} = (CW)_{tj} + \varepsilon_{tj}, \qquad \varepsilon_{tj} \sim \mathcal N(0, \sigma^2)$$
So $X_{tj}$ is drawn from a bell curve **centred on the rebuild**, $\mu = (CW)_{tj}$:
$$p(X_{tj} \mid C, W) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\Big(-\frac{\big(X_{tj} - (CW)_{tj}\big)^2}{2\sigma^2}\Big)$$
**This is where the square comes from:** the $(x - \mu)^2$ of the bell curve, with $x$ = what you recorded and
$\mu$ = what the model predicts. It is your residual, squared.

**(c) From one entry to all of them.** If the noise of different entries is independent, the probability of
the whole recording is the **product** of the T × 16 densities. Products are awkward, so take the log, which turns
a product into a sum and $\log(\exp(z)) = z$:
$$\log p(X \mid C, W) = \sum_{t,j}\Big[-\tfrac12\log(2\pi\sigma^2) - \frac{(X_{tj} - (CW)_{tj})^2}{2\sigma^2}\Big]
= \text{const} - \frac{1}{\sigma^2}\cdot\underbrace{\tfrac12\sum_{t,j}\big(X_{tj} - (CW)_{tj}\big)^2}_{L}$$
The constant doesn't depend on `C` or `W`. So the `C, W` that make the recording **most probable** (largest
log-probability, the *maximum likelihood* estimate) are exactly the ones with the **smallest L**.

**(d) In words.** Minimising squared error = choosing the synergies under which the EMG you actually recorded is
least surprising, **if** the noise is a bell curve of the same width everywhere.

**(e) The hidden assumptions, and whether EMG meets them:**

| assumption inside "squared error" | true for EMG envelopes? |
|---|---|
| noise is symmetric and can be negative | not quite: envelopes are ≥ 0 |
| the same spread σ on every electrode and frame | usually not: a louder channel is noisier |
| the noise at different entries is independent | no: frames 20 ms apart are strongly correlated |

So squared error is a sensible default, not the "true" loss for EMG. Change the noise model and the loss changes:
noise that grows with the signal (Poisson) gives the **KL divergence** (you build it in step 1f), and heavy-tailed
noise (Laplace) gives the absolute error from reason 1. **Every loss is a noise model in disguise.**

#### 3 · It is smooth, so it has a gradient everywhere

| loss per entry | its slope with respect to $r$ | at $r = 0$ |
|---|---|---|
| $\tfrac12 r^2$ | $r$ | 0, smoothly |
| $\lvert r\rvert$ | −1 if $r < 0$, +1 if $r > 0$ | **undefined**: a sharp corner where the slope jumps |

What that buys:
- **A gradient everywhere**, so step 1b's formulas are always defined and `numerical_grad` can check them. At a
  corner, a central difference averages −1 and +1 into a meaningless 0.
- **The push shrinks near the answer.** For $\tfrac12 r^2$ the slope is $r$ itself: a big miss gives a big push and
  a small miss a small one, so gradient descent slows down by itself near the bottom. With $\lvert r\rvert$ the push is
  always size 1, so a fixed step keeps overshooting.
- **A bowl in each matrix** (next cell), the shape that steps 1c and 1d rely on.

#### 4 · The ½
- **It makes the gradient clean.** $\frac{d}{dr}\big(\tfrac12 r^2\big) = r$, where $\frac{d}{dr}\,r^2 = 2r$. With the ½, the gradient
  is $C^\top(CW - X)$; without it, a 2 appears in every formula.
- **It doesn't move the minimum.** Multiplying by a positive number keeps the order of all values: if
  $L(a) < L(b)$ then $\tfrac12 L(a) < \tfrac12 L(b)$. The best `C, W` stays the best.
- **But it does change step sizes.** Without the ½ the gradient doubles, so the same learning rate takes twice the
  step (step 1c). "η = 1e-3" only means something once you know the convention. (The multiplicative updates in
  step 1d don't care: the 2 would appear on the top and the bottom and cancel.)
- **It also falls out of the Gaussian.** With σ = 1, the log-probability in reason 2 is
  $\text{const} - \tfrac12\sum(X - CW)^2$: the ½ is already there. The 2 in the bell curve's $2\sigma^2$ is the same 2
  that differentiation brings down.

### What makes NMF hard: one bottom vs a curved valley
**A bowl** is a function with exactly **one bottom** and walls that curve up in every direction, like
$f(w) = (w - 3)^2$. Bowls are easy: walk downhill from anywhere and you reach the bottom; often there is even a formula
for it. (The technical name is a *convex* function.)

Take the smallest NMF there is: one number for `C`, one for `W`, and data $x = 1$:
$$L(c, w) = \tfrac12\,(1 - c\,w)^2$$

**Fix one, and it is a bowl in the other.** Fix $c = 2$: $L(w) = \tfrac12(1 - 2w)^2$ is a parabola in $w$ with one
bottom, at $w = 0.5$. The same holds for any fixed $c$, and for real matrices. With `C` fixed, $L$ is a bowl in `W`
whose curvature is $C^\top C$ (the matrix behind step 1c's safe step); with `W` fixed, it is a bowl in `C`.

**Let both move, and it is not a bowl.** The perfect answers are **every** pair with $c\,w = 1$: (1, 1), (2, 0.5),
(4, 0.25), … So instead of one bottom there is a **curved valley floor** (the curve $w = 1/c$). A bowl can't have
that. In a bowl, the point halfway between two lowest points is also a lowest point. Check:

| point $(c, w)$ | $c\,w$ | $L$ |
|---|---|---|
| (1, 1) | 1 | **0**, perfect |
| (2, 0.5) | 1 | **0**, perfect |
| halfway: (1.5, 0.75) | 1.125 | **0.008**, worse |

The straight line between two perfect answers **climbs out of the valley**. Without the ≥ 0 rule it is worse
still: (1, 1) and (−1, −1) are both perfect, but halfway between them, (0, 0), has $L = 0.5$, a hump between two
separate valleys.

With real matrices there are more separate valleys still: relabelling the synergies (calling synergy 2 "synergy 1")
gives the same loss somewhere else entirely, and there can be genuine **local minima**, low spots that are not the
lowest.

**What this means in practice:**
1. **There is no formula for the answer**, but each half is easy. So NMF **alternates**: fix `C` and improve `W`
   (a bowl), then fix `W` and improve `C` (a bowl). That is exactly one round of `nmf_fit` in step 1e.
2. **Different starts can end in different places.** The answer depends on the random start, so the notebook fixes
   `seed=SEED`, and stage 2 asks whether the synergies stay the same across starts.
3. **The scale and order of synergies are not determined** (the valley floor, and the relabelling), so comparing
   two fits needs normalising and matching first (Q1a (b), and `match_synergies` in step 1e).

*Optional, to see it:* draw $L(c, w) = \tfrac12(1 - cw)^2$ as a contour plot over $c, w \in [0, 3]$ (`np.meshgrid`,
`plt.contourf`), and add the line from (1, 1) to (2, 0.5). The valley floor is the curve $w = 1/c$.

✍️ **Q1a.** **(a)** The loss of rebuilding everything as 0 is just $\tfrac12\sum X^2$. Why is that a natural "worst
sensible answer" to compare against? (Task 2 turns exactly this comparison into VAF.) **(b)** Write down two
*different* pairs `(C, W)` with the same loss, using only a positive number `s`. What does this mean for
comparing the synergies of two fits?

> *Your answer:* That assumes the residual is maximum from zero. This assumes no synergies and therefore any fitted model should perform better. Therefore it can be used as a benchmark. Two different pairs with the same loss can be defined as long as their multiples =CW at the end. So like 1/4, and 4 for example. This means that are many possible solutions that yield the same loss that contain combinations of different C and W. Would we need more analysis on individual C or W? We actually ndo no need more analysis bur rather normalization and mwatching. This is explored in 1e

### Step 1b · The gradient: which way is downhill?
The gradient tells us, for every entry of `W` (and of `C`), how fast `L` rises if we nudge that entry up.
Walking against it lowers the loss. Derive it with the chain rule. Call the error the **residual**,
$R = CW - X$ (shape T × D), so $L = \tfrac12\sum_{t,j} R_{tj}^2$.

**For one entry $W_{ij}$** (synergy *i*, electrode *j*): it only appears in column *j* of the rebuild,
$(CW)_{tj} = \sum_{i'} C_{ti'}W_{i'j}$, and $\partial (CW)_{tj}/\partial W_{ij} = C_{ti}$. So

$$\frac{\partial L}{\partial W_{ij}} = \sum_t R_{tj}\,\frac{\partial (CW)_{tj}}{\partial W_{ij}}
= \sum_t C_{ti}\,R_{tj} = \big(C^\top R\big)_{ij}
\quad\Longrightarrow\quad \nabla_W L = C^\top (CW - X) = C^\top C\,W - C^\top X$$

**For one entry $C_{ti}$** the same steps, with the roles swapped, give

$$\nabla_C L = (CW - X)\,W^\top = C\,W W^\top - X W^\top$$

**Read the two halves of $\nabla_W L = C^\top C W - C^\top X$.** Both are non-negative (products of
non-negative numbers).
- $C^\top X$ is **what the data asks for**: how strongly the activation of synergy *i* goes along with
  electrode *j*'s real activity.
- $C^\top C W$ is **what the model already gives**: the same, measured on the rebuilt data.

If the model gives more than the data asks, the gradient is positive and the entry should shrink. Step 1d builds
the whole algorithm on this "asks vs gives" split.

**Checking a gradient numerically.** You can check any derivative without doing calculus. Nudge one entry
by a tiny `h` both ways and measure the slope: $\partial L/\partial W_{ij} \approx \big(L(W_{ij}+h) -
L(W_{ij}-h)\big)/2h$. This *central difference* is the checker. You write it **first** (`numerical_grad`, next cell), test it on
functions whose slope you already know, and only then use it to test your `nmf_grads`. Do this whenever you
derive a gradient by hand.

#### Step 1b, part 1 · Build the checker first: `numerical_grad`
For a function `f` that takes an array `A` and returns one number, `numerical_grad` returns an array shaped like
`A`. Each entry is the slope of `f` when you nudge **that one entry** of `A` and hold the rest fixed:

$$g_{\text{idx}} \;\approx\; \frac{f(A + h\,e_{\text{idx}}) - f(A - h\,e_{\text{idx}})}{2h}$$

where $e_{\text{idx}}$ is all zeros except a 1 at `idx`.

**Why central, not forward** $\big(f(A+h) - f(A)\big)/h$? Expand with Taylor:
$f(a \pm h) = f(a) \pm h f'(a) + \tfrac{h^2}{2} f''(a) \pm \tfrac{h^3}{6} f'''(a) + \dots$
- Forward difference: $f' + \tfrac{h}{2} f'' + \dots$, so its error shrinks like $h$.
- Central difference: the $f''$ terms cancel, leaving $f' + \tfrac{h^2}{6} f''' + \dots$, so its error shrinks like $h^2$.
  With $h = 10^{-6}$ that is about $10^{-12}$ instead of $10^{-6}$.

**Why $h = 10^{-6}$ and not $10^{-12}$?** A float64 keeps about 16 significant digits (machine ε ≈ 2·10⁻¹⁶).
$f(A+h)$ and $f(A-h)$ are almost equal, and subtracting two almost-equal numbers wipes out their shared digits:
the **round-off** error is about $ε\,|f| / h$, which *grows* as $h$ shrinks. The Taylor error shrinks with $h$, the
round-off error grows; for central differences the best trade-off is near $h \approx ε^{1/3} \approx 10^{-5}$–$10^{-6}$.

**Why test the checker?** If `numerical_grad` is wrong, a wrong `nmf_grads` could pass. So its own check uses
functions whose slope you know by hand: $\sum A^2$ (slope $2A$), $\sum B \odot A$ (slope $B$), and $\sum A^3$
(slope $3A^2$; this one fails a forward difference).

In [52]:
from collections.abc import Callable


def numerical_grad(f: Callable[[np.ndarray], float], A: np.ndarray, h: float = 1e-6) -> np.ndarray:
    """Numerical gradient of a scalar function f at A, by central differences, one entry at a time.

    Parameters
    ----------
    f : function taking an array shaped like A and returning a float
    A : float array of any shape, the point where the slope is measured (NOT changed)
    h : float, the size of the nudge

    Returns
    -------
    g : float array shaped like A, with g[idx] ≈ (f(A with A[idx] + h) - f(A with A[idx] - h)) / (2h)
    """
    # TODO
    
    g = np.zeros(A.shape)
    for i in np.ndindex(A.shape):
        Ap, Am = A.copy(),A.copy()
        Ap[i]+= h
        Am[i]-= h
        g[i] = (f(Ap) - f(Am))/(2*h)
    return g
    # raise NotImplementedError

#### 💡 Stuck on `numerical_grad`? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Make an output array of zeros, shaped like `A`, of **float** type.
2. Visit every index of `A` (it should work for 1-D, 2-D, any number of dimensions).
3. At each index: make two copies of `A`, add `h` to that entry in one and subtract it in the other, call `f`
   on both, and store the difference divided by `2h`.
4. Never change `A` itself: the caller still needs it.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.ndindex(A.shape)` yields every index as a tuple, whatever the number of dimensions:
  `list(np.ndindex((2, 2)))` → `[(0, 0), (0, 1), (1, 0), (1, 1)]`. A tuple indexes one entry: `A[(1, 0)]`.
- `A.copy()` gives a new array; changing it leaves `A` alone. Writing `Ap = A` does **not** copy.
- `np.zeros(A.shape)` is float. `np.zeros_like(A)` copies `A`'s type: for an integer `A` it would round every
  slope to a whole number without any error.

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def numerical_grad(f, A, h=1e-6):
    g = np.zeros(____)
    for i in np.ndindex(____):
        Ap, Am = A.copy(), A.copy()
        Ap[i] += ____
        Am[i] -= ____
        g[i] = (f(Ap) - f(____)) / (____)
    return g
```

</details>

In [ ]:
# ✅ Check step 1b, part 1: test the checker on functions whose slope you know
rng_g = np.random.default_rng(2)
A_g, B_g = rng_g.random((3, 4)), rng_g.random((3, 4))
A_keep = A_g.copy()
g = numerical_grad(lambda A: float(np.sum(A ** 2)), A_g)
show("slope of sum(A²): should be 2A", g, 2 * A_g)
show("slope of sum(B * A): should be B", numerical_grad(lambda A: float(np.sum(B_g * A)), A_g), B_g)
assert np.array_equal(A_g, A_keep), "numerical_grad changed A: nudge copies, not A itself"
assert isinstance(g, np.ndarray) and g.shape == A_g.shape, \
    f"return an array shaped like A {A_g.shape} (got {getattr(g, 'shape', type(g).__name__)})"
assert np.allclose(g, 2 * A_g, atol=1e-6), "the slope of sum(A²) should be 2A"
assert np.allclose(numerical_grad(lambda A: float(np.sum(B_g * A)), A_g), B_g, atol=1e-6), \
    "the slope of sum(B * A) should be B"
g3 = numerical_grad(lambda A: float(np.sum(A ** 3)), A_g)
show("largest error on the slope of sum(A³)  (central ≈ 1e-10, forward ≈ 1e-6)", float(np.max(np.abs(g3 - 3 * A_g ** 2))))
assert np.allclose(g3, 3 * A_g ** 2, atol=1e-8), \
    f"the slope of sum(A³) is off by {np.abs(g3 - 3 * A_g ** 2).max():.1e}: a forward difference is off by about 3·A·h; use the central one"
v_g = np.linspace(0, 3, 5)
assert np.allclose(numerical_grad(lambda v: float(np.sum(np.sin(v))), v_g), np.cos(v_g), atol=1e-8), \
    "it should work on 1-D arrays too: loop over np.ndindex(A.shape)"
print("✅ numerical_grad works: it gives 2A, B and 3A² for the three test functions, and cos for sin")

In [54]:
def nmf_grads(X: np.ndarray, C: np.ndarray, W: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Gradients of nmf_loss with respect to C and W.

    Parameters
    ----------
    X : (T, D), C : (T, k), W : (k, D), as in nmf_loss

    Returns
    -------
    gC : (T, k) float, dL/dC = (C W - X) W.T
    gW : (k, D) float, dL/dW = C.T (C W - X)
    Each gradient has the same shape as the matrix it belongs to.
    """
    # TODO
    gC = (C @ W - X) @ W.T
    gW = C.T @ (C @ W - X)
    return gC,gW

In [57]:
rng = np.random.default_rng(SEED)
k = 4
C_mine = rng.random((X_sub.shape[0], k))   # (20000, 4), values between 0 and 1
W_mine = rng.random((k, X_sub.shape[1]))   # (4, 16)
nmf_grads(X_sub, C0, W0)[1] #Weights

array([[ -839.48530611,  -520.21033757,  -267.10797538,  -291.30626828,
         -536.99032457,  -477.17987229,  -305.50904749,  -621.82425144,
         -794.9416386 , -1033.9373059 , -1330.04994557, -1097.1261948 ,
        -1212.1669758 ,  -923.12926533,  -778.17542502,  -458.4706076 ],
       [ -828.03064137,  -543.48299272,  -258.39050057,  -306.08966654,
         -520.00120503,  -478.22304865,  -306.29029051,  -626.01590169,
         -785.52430984, -1039.87235796, -1331.15867181, -1105.93077858,
        -1201.96915262,  -923.5500406 ,  -767.61279484,  -469.70252348],
       [ -836.89022711,  -533.34288298,  -256.56014801,  -295.12086851,
         -550.39794504,  -464.55626281,  -325.87691269,  -603.9262972 ,
         -779.31103966, -1047.64350257, -1317.14754133, -1115.03167014,
        -1180.98653898,  -927.70902115,  -767.60694596,  -466.17567986],
       [ -836.03473607,  -511.76292701,  -273.39933738,  -302.3693378 ,
         -540.26051283,  -474.60876341,  -333.68287022,  -638

#### 💡 Stuck on step 1b? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Compute the residual once: `R = C @ W - X`, shape (T, D).
2. `gC` must have the shape of `C`, (T, k): multiply `R` on the right by `W` turned on its side.
3. `gW` must have the shape of `W`, (k, D): multiply `R` on the left by `C` turned on its side.
4. Return them in the order `(gC, gW)`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `A.T` is the transpose. Let the shapes guide you: (T, D) @ (D, k) → (T, k); (k, T) @ (T, D) → (k, D).
- The sign matters: the residual is *model minus data*, `C @ W - X`, not `X - C @ W`.

```python
R = np.ones((3, 2)); W = np.ones((4, 2))
(R @ W.T).shape        # (3, 4)
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def nmf_grads(X, C, W):
    R = ____ @ ____ - ____
    gC = R @ ____.T
    gW = ____.T @ R
    return gC, gW
```

</details>

In [ ]:
# ✅ Check step 1b, part 2: compare your formula with the numerical slope from YOUR numerical_grad (tested above)
gC, gW = nmf_grads(X_t, C_t, W_t)
assert gC.shape == C_t.shape and gW.shape == W_t.shape, \
    f"shapes {gC.shape}, {gW.shape}; expected {C_t.shape} (like C) and {W_t.shape} (like W). Wrong order?"
num_gW = numerical_grad(lambda W_: ref.nmf_loss(X_t, C_t, W_), W_t)
num_gC = numerical_grad(lambda C_: ref.nmf_loss(X_t, C_, W_t), C_t)
show("dL/dW: your formula vs the numerical slope", gW, num_gW)
show("dL/dC: your formula vs the numerical slope", gC, num_gC)
assert not np.allclose(gW, -num_gW, atol=1e-5), "dL/dW has the wrong sign: the residual is C @ W - X"
assert np.allclose(gW, num_gW, atol=1e-5), f"dL/dW is off by up to {np.abs(gW - num_gW).max():.2e}"
assert np.allclose(gC, num_gC, atol=1e-5), f"dL/dC is off by up to {np.abs(gC - num_gC).max():.2e}"
print(f"✅ Your gradients match the numerical slopes to {max(np.abs(gW - num_gW).max(), np.abs(gC - num_gC).max()):.1e}")

✍️ **Q1b.** **(a)** At the random start (`C0`, `W0`), compute `nmf_grads(X_sub, C0, W0)[1]`. Is the gradient for `W` mostly
positive or mostly negative, and what does that say about the random start: is its rebuild too big or too small?
**(b)** Where the gradient for an entry is exactly 0, nudging that entry doesn't change the loss (to first order).
Why might a non-negative entry still *not* be at its best value when its gradient is **positive** and the entry
is already 0?

> *Your answer:*

the gradients appear mostly positive and very large. The bruild is too big perhapps. The frames are also all added up so there is not scaling.  I am not sure how to intepret this tho- it seems that this is the unscaled version. These large positive gradients mean that increasing the weight makes the loss worse, so our goal is to shrink the weights.

In the scaled version, we have mostly negative weights, this means we might need to grow the weights which will decrease loss as our rebuild is too small. Also we can see the synergies, they all seem relatively propotional. 

### Step 1c · Plain gradient descent, and why it struggles
The obvious algorithm: step a little against the gradient, then push anything that went negative back to 0
(**projected gradient descent**; "projected" means "moved back into the allowed region"):

$$W \leftarrow \max\!\big(0,\; W - \eta\,\nabla_W L\big) \qquad\text{then}\qquad
C \leftarrow \max\!\big(0,\; C - \eta\,\nabla_C L\big)$$

Always update the two in turn, using the **new** `W` when you compute the gradient for `C`. This is
*alternating* (block coordinate) descent: each half-step solves a simpler, bowl-shaped problem.

**The catch is the learning rate η.** For a bowl-shaped function, a step is only safe if η is below about
$1/\lambda_{\max}$, where $\lambda_{\max}$ is the largest **curvature**: the largest eigenvalue of the second
derivative. Go above roughly $2/\lambda_{\max}$ and the steps overshoot further each time, so the loss
explodes. Differentiate $\nabla_W L = C^\top C W - C^\top X$ once more and you get $C^\top C$, so:
- the curvature for `W` is set by $C^\top C$, a sum over **all T = 20,000 frames**, so it is huge;
- the curvature for `C` is set by $W W^\top$, a sum over **16 electrodes only**, so it is small.

**Prediction before you run it:** can a single η suit both matrices?

In [71]:
def pgd_step(X: np.ndarray, C: np.ndarray, W: np.ndarray, lr: float) -> tuple[np.ndarray, np.ndarray]:
    """One round of projected gradient descent, with the same learning rate for both matrices.

    1. W_new = max(0, W - lr * dL/dW)                  (gradient at the current C, W)
    2. C_new = max(0, C - lr * dL/dC)                  (gradient at C, W_new: use the NEW W)

    Parameters
    ----------
    X : (T, D), C : (T, k), W : (k, D), as before
    lr : float, the learning rate (step size) η

    Returns
    -------
    (C_new, W_new): new arrays (don't change the ones passed in), both ≥ 0
    """
    # TODO
    # raise NotImplementedError
    _, dW = nmf_grads(X,C,W)
    W_new = np.maximum(0, W - lr * dW)
    
    dC, _ = nmf_grads(X,C,W_new)
    C_new = np.maximum(0,C - lr * dC)
    return C_new, W_new

#### 💡 Stuck on step 1c? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Call your `nmf_grads(X, C, W)` and keep the gradient for `W`. Take the step, then clip at 0.
2. Call `nmf_grads` **again**, now with the new `W`, and keep the gradient for `C`. Step and clip.
3. Return `(C_new, W_new)`, in that order.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.maximum(0.0, A)` replaces every negative entry of `A` by 0 and returns a new array.
- `_, gW = nmf_grads(...)` keeps only the second value.

```python
np.maximum(0.0, np.array([0.5, -0.2, 1.0]))   # array([0.5, 0. , 1. ])
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def pgd_step(X, C, W, lr):
    _, gW = nmf_grads(X, C, W)
    W_new = np.maximum(0.0, W - ____ * ____)
    gC, _ = nmf_grads(X, C, ____)
    C_new = np.maximum(0.0, ____ - lr * ____)
    return C_new, W_new
```

</details>

In [ ]:
# ✅ Check step 1c
C_keep, W_keep = C_t.copy(), W_t.copy()
C1, W1 = pgd_step(X_t, C_t, W_t, 0.05)
assert np.array_equal(C_t, C_keep) and np.array_equal(W_t, W_keep), "pgd_step changed its inputs: return new arrays"
assert C1.shape == C_t.shape and W1.shape == W_t.shape, "return (C_new, W_new) in that order"
assert (C1 >= 0).all() and (W1 >= 0).all(), "clip both matrices at 0 after the step"
r_C1, r_W1 = ref.pgd_step(X_t, C_t, W_t, 0.05)
show("W_new after one step (η = 0.05)", W1, r_W1)
show("loss before → after that step", f"{ref.nmf_loss(X_t, C_t, W_t):.4f} → {ref.nmf_loss(X_t, C1, W1):.4f}")
assert np.allclose(W1, r_W1), "W_new differs from the reference: step against the gradient, then clip"
assert np.allclose(C1, r_C1), "C_new differs: compute the gradient for C with the NEW W"
assert ref.nmf_loss(X_t, C1, W1) < ref.nmf_loss(X_t, C_t, W_t), "a small step should lower the loss"
C_big, W_big = pgd_step(X_t, C_t, W_t, 5.0)                 # a huge step pushes entries below 0 ...
show("smallest entry after a huge step (η = 5): clipping keeps it ≥ 0", float(min(C_big.min(), W_big.min())))
assert (C_big >= 0).all() and (W_big >= 0).all(), "after a big step some entries are negative: clip at 0"
assert np.allclose(W_big, ref.pgd_step(X_t, C_t, W_t, 5.0)[1]), "with a big step W_new differs from the reference"
print("✅ pgd_step works")

### 📊 Experiment: one learning rate for both matrices
**The largest safe step.** Hold `C` fixed. As a function of `W`, $L = \tfrac12\lVert X - CW\rVert_F^2$ is a bowl,
and the matrix that sets how sharply it curves (its second derivative, the *Hessian*, for each column of `W`) is
$C^\top C$. Along a direction with curvature $\lambda$, one gradient step of size η multiplies the distance to the
bottom by $(1 - \eta\lambda)$:
- if $\eta\lambda < 1$ you move part of the way down (slow when η is tiny);
- if $1 < \eta\lambda < 2$ you overshoot but still land closer;
- if $\eta\lambda > 2$ you overshoot **further** each round, and the loss blows up.

The steepest direction has curvature $\lambda_{\max}(C^\top C)$, so $\eta = 1/\lambda_{\max}(C^\top C)$ is the
**safe step for `W`**: it never overshoots in any direction. With the roles swapped, the safe step for `C` is
$1/\lambda_{\max}(W W^\top)$.

**You write:**
- `safe_steps(C, W)` → `(safe_W, safe_C)`
- `run_pgd(X, C, W, lr, n_rounds)` → the loss after each of `n_rounds` rounds of your `pgd_step`

Then, in the same cell: print the two safe steps at the random start (`C0`, `W0`) and how many times bigger the
one for `C` is; run 200 rounds from `C0, W0` for each η in `[1e-5, 1e-4, 1e-3, 5e-3]` into `pgd_curves`; print each
final loss; and plot every curve (log y axis). **What to look for:** how far apart the two safe steps are, which η
makes progress, and what happens to the largest one.

In [ ]:
def safe_steps(C: np.ndarray, W: np.ndarray) -> tuple[float, float]:
    """Largest safe gradient step for each matrix: 1 / (largest eigenvalue of its curvature matrix).

    Parameters
    ----------
    C : (T, k), W : (k, D), as before

    Returns
    -------
    (safe_W, safe_C) : plain floats, 1 / λmax(Cᵀ C) and 1 / λmax(W Wᵀ)
    """
    # TODO
    raise NotImplementedError


def run_pgd(X: np.ndarray, C: np.ndarray, W: np.ndarray, lr: float, n_rounds: int) -> list[float]:
    """Run n_rounds of your pgd_step from (C, W) and record the loss.

    Returns
    -------
    losses : list of n_rounds floats, nmf_loss after each round. The C and W passed in are NOT changed.
    """
    # TODO
    raise NotImplementedError


N_ROUNDS = 200
safe_W, safe_C = None, None                         # TODO: safe_steps at the random start C0, W0
pgd_curves: dict[float, list[float]] = {}           # learning rate → loss after each round (step 1e reuses it)
with np.errstate(over="ignore", invalid="ignore"):  # the largest rate is meant to blow up
    for lr in [1e-5, 1e-4, 1e-3, 5e-3]:
        ...                                         # TODO: pgd_curves[lr] = run_pgd(... from C0, W0 on X_sub ...)
# TODO: print safe_W, safe_C and safe_C / safe_W; print the final loss for each lr
# TODO: plot each curve against round number, one line per η with a legend, log y axis, axis labels and a title.
#       Clip the values at 1e7 first, so the curve that blows up stays on the plot.

#### 💡 Stuck on the experiment? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `safe_steps`: build the two small k × k matrices $C^\top C$ and $W W^\top$, find the largest eigenvalue of each,
   and return 1 over each, as floats.
2. `run_pgd`: start an empty list; repeat `n_rounds` times: one `pgd_step`, then append the loss. Because
   `pgd_step` returns **new** arrays, the caller's `C` and `W` are never touched.
3. The experiment: one `run_pgd` per learning rate, each starting from `C0, W0`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.linalg.eigvalsh(M)` gives the eigenvalues of a **symmetric** matrix (like $C^\top C$), smallest first.
  `.max()` picks the largest.
- `np.clip(curve, None, 1e7)` caps values at 1e7 (no lower cap).
- `ax.set_yscale("log")`; `ax.plot(y, label=f"η = {lr:.0e}")`; `ax.legend()`.

```python
np.linalg.eigvalsh(np.array([[2.0, 0.0], [0.0, 5.0]]))   # array([2., 5.])
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def safe_steps(C, W):
    return float(1 / np.linalg.eigvalsh(____).max()), float(1 / np.linalg.eigvalsh(____).max())

def run_pgd(X, C, W, lr, n_rounds):
    losses = []
    for _ in range(n_rounds):
        C, W = pgd_step(X, C, W, ____)
        losses.append(nmf_loss(____, C, W))
    return losses

safe_W, safe_C = safe_steps(____, ____)
...
        pgd_curves[lr] = run_pgd(X_sub, ____, ____, lr, N_ROUNDS)

fig, ax = plt.subplots(figsize=(6, 3.5))
for lr, curve in pgd_curves.items():
    ax.plot(np.clip(curve, None, ____), label=f"η = {lr:.0e}")
ax.set_yscale(____); ax.legend(); plt.show()
```

</details>

In [ ]:
# ✅ Check step 1c experiment
sW, sC = safe_steps(C_t, W_t)
rW, rC = ref.safe_steps(C_t, W_t)
show("(safe_W, safe_C) on the tiny problem", np.array([sW, sC]), np.array([rW, rC]))
assert isinstance(sW, float) and isinstance(sC, float), "return plain floats: wrap each in float(...)"
assert np.isclose(sW, rW) and np.isclose(sC, rC), \
    f"got ({sW:.3g}, {sC:.3g}), expected ({rW:.3g}, {rC:.3g}): safe_W uses C.T @ C, safe_C uses W @ W.T; 1 / the LARGEST eigenvalue"
C_keep, W_keep = C_t.copy(), W_t.copy()
L_run = run_pgd(X_t, C_t, W_t, 0.05, 5)
show("loss after each of 5 PGD rounds (η = 0.05)", np.array(L_run), np.array(ref.run_pgd(X_t, C_t, W_t, 0.05, 5)))
show("safe steps at the random start (safe_W, safe_C)", "not set yet" if safe_W is None else np.array([safe_W, safe_C]),
     np.array(ref.safe_steps(C0, W0)))
show("final loss for each η", ", ".join(f"η = {lr:.0e}: {(c[-1] if c else float('nan')):,.0f}" for lr, c in pgd_curves.items())
     or "pgd_curves is empty")
assert np.array_equal(C_t, C_keep) and np.array_equal(W_t, W_keep), "run_pgd changed its inputs"
assert len(L_run) == 5, f"{len(L_run)} losses for 5 rounds: record one loss after each round"
assert np.allclose(L_run, ref.run_pgd(X_t, C_t, W_t, 0.05, 5)), "losses differ: one pgd_step, THEN nmf_loss, each round"
assert safe_W is not None and np.isclose(safe_W, ref.safe_steps(C0, W0)[0]), "safe_W, safe_C: safe_steps(C0, W0)"
assert sorted(pgd_curves) == [1e-5, 1e-4, 1e-3, 5e-3] and all(len(c) == N_ROUNDS for c in pgd_curves.values()), \
    "pgd_curves needs one 200-long curve for each of the 4 learning rates"
print("✅ safe_steps and run_pgd work, and pgd_curves is complete")

✍️ **Q1c.** **(a)** Explain the four curves with the two safe steps. Why is the small η slow, and why does the large η blow
up? Which matrix limits η, and why (think about what $C^\top C$ adds up)? **(b)** Could you fix this with
**two** learning rates, one per matrix? Why would that still not be enough, given that the safe steps keep
changing as `C` and `W` change during the fit? **(c)** What does the clipping at 0 do to an entry once it is 0
and its gradient is positive?

> *Your answer:*

### Step 1d · Multiplicative updates: a step size for every entry
Lee & Seung's (2001) idea: don't pick one η. Give **every entry its own step**, chosen so that the update
can never cross 0 and never needs tuning. Write the gradient step for `W` with an entry-by-entry step `H`:

$$W \;\leftarrow\; W - H \odot \nabla_W L \;=\; W - H \odot \big(C^\top C W - C^\top X\big)$$

Now choose $\;H = \dfrac{W}{C^\top C\,W}\;$ (entry by entry). Substitute and simplify:

$$W - \frac{W}{C^\top C W}\odot C^\top C W \;+\; \frac{W}{C^\top C W}\odot C^\top X
\;=\; W - W + W \odot \frac{C^\top X}{C^\top C\,W}
\;=\; \boxed{\,W \odot \frac{C^\top X}{C^\top C\,W}\,}$$

The same choice for `C`, $H_C = C/(C W W^\top)$, gives

$$\boxed{\,C \leftarrow C \odot \frac{X W^\top}{C\,W W^\top}\,}$$

**What this buys:**
1. **Never negative, no clipping.** Every number in the boxed formulas is ≥ 0, so a product of them is too.
2. **No learning rate.** The step `H` sets itself, entry by entry, from the current `C` and `W`. Large entries
   take large steps and entries near 0 take tiny ones, so nothing overshoots past 0.
3. **"Asks vs gives".** The ratio is step 1b's two halves of the gradient: what the data asks for, divided by
   what the model already gives. Ratio > 1 → the entry grows; < 1 → it shrinks; = 1 (gradient 0) → it stays.
4. **Symmetry.** Transposing `X ≈ C W` gives `Xᵀ ≈ Wᵀ Cᵀ`: the same problem with the roles of `C` and `W`
   swapped. So the `C` update is the `W` update applied to the transposed problem. The check tests this.

In code, add a tiny `EPS` to every denominator so an all-zero row or column never divides by 0.

In [ ]:
EPS = 1e-10   # added to every denominator so nothing divides by 0


def update_W(X: np.ndarray, C: np.ndarray, W: np.ndarray) -> np.ndarray:
    """Multiplicative update of the synergies: W ⊙ (Cᵀ X) / (Cᵀ C W + EPS).

    Parameters
    ----------
    X : (T, D), C : (T, k), W : (k, D), as before

    Returns
    -------
    W_new : (k, D) float ≥ 0 (a new array)
    """
    # TODO
    raise NotImplementedError


def update_C(X: np.ndarray, C: np.ndarray, W: np.ndarray) -> np.ndarray:
    """Multiplicative update of the activations: C ⊙ (X Wᵀ) / (C W Wᵀ + EPS).

    Returns
    -------
    C_new : (T, k) float ≥ 0 (a new array)
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on step 1d? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Each function is one line: the current matrix, times (entry by entry) a top part divided by a bottom part.
2. `update_W`: top = what the data asks for, `Cᵀ X` (k, D); bottom = what the model gives, `Cᵀ C W` (k, D).
3. `update_C`: top = `X Wᵀ` (T, k); bottom = `C W Wᵀ` (T, k).
4. Add `EPS` to the bottom.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `@` for the matrix products inside the top and bottom; `*` and `/` between the three (k, D) arrays.
- Write `W * top / bottom`, not `W *= ...`: the in-place version would change the caller's `W`.
- Order matters for speed, not for the answer: `C.T @ C @ W` computes the small (k, k) matrix `C.T @ C` first.

```python
W = np.array([[2.0]]); top = np.array([[3.0]]); bottom = np.array([[6.0]])
W * top / bottom        # [[1.]]: the model gave twice what the data asked for, so W halves
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def update_W(X, C, W):
    return W * (____.T @ ____) / (____.T @ ____ @ ____ + EPS)


def update_C(X, C, W):
    return C * (____ @ ____.T) / (____ @ ____ @ ____.T + EPS)
```

</details>

In [ ]:
# ✅ Check step 1d
C_keep, W_keep = C_t.copy(), W_t.copy()
W1, C1 = update_W(X_t, C_t, W_t), update_C(X_t, C_t, W_t)
show("W after one update_W", W1, ref.update_W(X_t, C_t, W_t))
show("loss: start → after update_W → after update_C (with the new W)",
     f"{ref.nmf_loss(X_t, C_t, W_t):.4f} → {ref.nmf_loss(X_t, C_t, W1):.4f} → "
     f"{ref.nmf_loss(X_t, update_C(X_t, C_t, W1), W1):.4f}")
assert np.array_equal(C_t, C_keep) and np.array_equal(W_t, W_keep), \
    "an update changed its input: write W * top / bottom, not W *= ..."
assert W1.shape == W_t.shape and C1.shape == C_t.shape, f"shapes {W1.shape}, {C1.shape}"
assert np.allclose(W1, ref.update_W(X_t, C_t, W_t)), "update_W differs from W * (C.T X) / (C.T C W)"
assert np.allclose(C1, ref.update_C(X_t, C_t, W_t)), "update_C differs from C * (X W.T) / (C W W.T)"

# 1. It IS gradient descent, with the step H = W / (C.T C W) for every entry
_, gW_t = ref.nmf_grads(X_t, C_t, W_t)
assert np.allclose(W1, W_t - W_t / (C_t.T @ C_t @ W_t) * gW_t)
# 2. Symmetry: the C update is the W update on the transposed problem
assert np.allclose(C1, update_W(X_t.T, W_t.T, C_t.T).T), "update_C should equal update_W(X.T, W.T, C.T).T"
# 3. Never negative, and the loss never goes up: 500 random problems, both updates
rng_m = np.random.default_rng(1)
for _ in range(500):
    Xr, Cr, Wr = rng_m.random((8, 6)), rng_m.random((8, 3)), rng_m.random((3, 6))
    Wn = update_W(Xr, Cr, Wr); Cn = update_C(Xr, Cr, Wn)
    assert (Wn >= 0).all() and (Cn >= 0).all()
    assert ref.nmf_loss(Xr, Cr, Wn) <= ref.nmf_loss(Xr, Cr, Wr) + 1e-12, "update_W raised the loss"
    assert ref.nmf_loss(Xr, Cn, Wn) <= ref.nmf_loss(Xr, Cr, Wn) + 1e-12, "update_C raised the loss"
H_W = W0 / (C0.T @ C0 @ W0)            # the step every entry of W takes at the random start
print(f"✅ Multiplicative updates work: they are gradient steps, symmetric, non-negative, and never raise the loss.\n"
      f"   Their steps for W at the start: {H_W.min():.1e} to {H_W.max():.1e}  (step 1c's safe step: {safe_W:.1e})")

<details>
<summary><b>Why can the loss never go up? (Lee & Seung's proof, in words, optional)</b></summary>

Fix `C` and look at one column `w` of `W` (one electrode). The loss is a bowl, $\ell(w) = \tfrac12\lVert x - Cw
\rVert^2$, with curvature matrix $A = C^\top C$. The proof replaces this bowl by a **steeper bowl that touches it at
the current point** and lies **above it everywhere else**. The steeper bowl uses the diagonal curvature
$K = \operatorname{diag}\big((Aw)_i / w_i\big)$ instead of $A$. Lee & Seung show $K - A$ is positive semi-definite
whenever `A` and `w` are non-negative, which is exactly what "above it everywhere" needs.

Then:
1. A diagonal bowl is easy to minimise exactly. One Newton step with curvature `K`, $w - K^{-1}\nabla\ell$, lands
   on its bottom, and that step *is* the multiplicative update.
2. The true loss there is at most the steeper bowl's value there (it lies above),
3. which is at most the steeper bowl's value at the start (we went to its bottom),
4. which equals the true loss at the start (they touch).

So the loss can only stay the same or go down. This trick is **majorise–minimise**: the same idea as the EM
algorithm. It guarantees descent, but **not** that we reach the best possible answer. Different random starts
can end in different local minima, which is why later notebooks compare seeds.
</details>

✍️ **Q1d.** **(a)** Using the boxed update, explain why an entry of `W` that is exactly 0 stays 0 forever. Why is a *random*
start therefore better than starting from zeros, and why must it be strictly positive? **(b)** The check printed
the range of steps `H` that the update takes for `W`. Compare it with step 1c's safe step. How does a per-entry step
solve the problem that a single η could not?

> *Your answer:*

### Step 1e · The whole algorithm, and a test on synergies we planted
Now put the pieces together.

**`nmf_init(X, k, seed)`: the random start.** `rng = np.random.default_rng(seed)`,
`scale = np.sqrt(X.mean() / k)`, then `C = scale * rng.random((T, k))` and `W = scale * rng.random((k, D))`, **in
that order**. Why that scale? Each rebuilt entry is a sum of `k` products of two numbers of size about `scale`, so
it is about `k × scale² = X.mean()` (a quarter of that on average, since `rng.random` averages 0.5): the same order
of magnitude as the data, so the first updates don't have to grow or shrink everything 1000-fold. Every entry is
strictly positive, which matters (Q1d). (Use exactly this recipe and your answer will match the reference.)

**`nmf_fit(X, k, n_iter, seed)`: the alternating loop.** Start with `nmf_init`, then repeat `n_iter` times:
`W ← update_W(X, C, W)`, then `C ← update_C(X, C, W)` with the new `W`. After each round, record
`nmf_loss(X, C, W)`.

**Tidy up the scale.** `C W` doesn't change if a row of `W` gets 10× bigger and the matching column of `C` 10×
smaller (step 1a, Q1a). So at the end scale every row of `W` to length 1 and multiply `C`'s columns by the old
lengths. Synergies from different fits then become comparable.

**`nmf_transform(X, W, n_iter, seed)`: new data, synergies fixed.** For test frames you keep `W` and only find `C`.
That means the same start for `C` (from `nmf_init`, ignoring its `W`), then `update_C` alone, repeated. With `W`
fixed the problem is a bowl in `C`, so this reaches the best `C` for those synergies.

**Then the real test of any method:** data where you **know the answer**. `ref.planted_synergies()` builds EMG
from 3 synergies we chose: bumps around the electrode ring, centred on electrodes 2, 7 and 12. Your `nmf_fit` gets
only `X`. Does it find them?

In [ ]:
def nmf_init(X: np.ndarray, k: int, seed: int) -> tuple[np.ndarray, np.ndarray]:
    """Random strictly positive start: C (T, k) then W (k, D), both scale * rng.random(...),
    with scale = sqrt(X.mean() / k) and rng = np.random.default_rng(seed)."""
    # TODO
    raise NotImplementedError


def nmf_fit(X: np.ndarray, k: int, n_iter: int = 300, seed: int = 0
            ) -> tuple[np.ndarray, np.ndarray, list[float]]:
    """Non-negative matrix factorisation X ≈ C @ W with multiplicative updates.

    Parameters
    ----------
    X      : (T, D) float, non-negative data
    k      : int, number of synergies
    n_iter : int, number of rounds (each round: update_W, then update_C)
    seed   : int, for the random start

    Returns
    -------
    C      : (T, k) float ≥ 0, activations
    W      : (k, D) float ≥ 0, synergies, each row scaled to length 1 (C scaled up to compensate)
    losses : list of n_iter floats, nmf_loss after each round
    """
    # TODO
    raise NotImplementedError


def nmf_transform(X: np.ndarray, W: np.ndarray, n_iter: int = 300, seed: int = 0) -> np.ndarray:
    """Activations for new data with the synergies held fixed.

    Parameters
    ----------
    X : (T, D) float, non-negative data
    W : (k, D) float, synergies from nmf_fit (NOT changed here)

    Returns
    -------
    C : (T, k) float ≥ 0, so that C @ W ≈ X. Start from nmf_init (keep its C), then repeat only update_C.
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on step 1e? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `nmf_init`: make the generator, the scale, then `C`, then `W`.
2. `nmf_fit`: `C, W = nmf_init(...)`; loop `n_iter` times: `W = update_W(...)`, `C = update_C(...)`,
   append `nmf_loss(...)`.
3. After the loop: compute the length of each row of `W`, divide the rows by it, multiply `C`'s columns by it.
4. `nmf_transform`: `C, _ = nmf_init(X, k, seed)` with `k = W.shape[0]`, then loop `update_C` only.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.linalg.norm(W, axis=1)` → the length of each row, shape (k,). `W / n[:, None]` divides row i by
  `n[i]`; `C * n` multiplies column i of `C` by `n[i]`.
- Add `EPS` to the lengths so a synergy that died out (all zeros) doesn't divide by 0.

```python
C = np.array([[1.0, 2.0]]); W = np.array([[3.0, 0.0], [0.0, 4.0]])
n = np.linalg.norm(W, axis=1)   # array([3., 4.])
(C * n) @ (W / n[:, None])      # [[3., 8.]]: the same as C @ W
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def nmf_init(X, k, seed):
    rng = np.random.default_rng(seed)
    scale = np.sqrt(X.mean() / ____)
    C = scale * rng.random((X.shape[0], ____))
    W = scale * rng.random((____, X.shape[1]))
    return C, W


def nmf_fit(X, k, n_iter=300, seed=0):
    C, W = nmf_init(X, k, seed)
    losses = []
    for _ in range(n_iter):
        W = update_W(X, ____, ____)
        C = update_C(X, ____, ____)
        losses.append(nmf_loss(X, C, W))
    n = np.linalg.norm(W, axis=____) + EPS
    return C * n, W / n[:, None], losses


def nmf_transform(X, W, n_iter=300, seed=0):
    C, _ = nmf_init(X, W.shape[0], seed)
    for _ in range(n_iter):
        C = update_C(X, ____, ____)
    return C
```

</details>

In [ ]:
# ✅ Check step 1e (about 10 s)
C_i, W_i = nmf_init(X_sub, 4, SEED)
r_Ci, r_Wi = ref.nmf_init(X_sub, 4, SEED)
show("nmf_init: first row of C, then first row of W", np.concatenate([C_i[0], W_i[0]]), np.concatenate([r_Ci[0], r_Wi[0]]))
assert np.allclose(C_i, r_Ci) and np.allclose(W_i, r_Wi), "nmf_init differs: draw C first, then W, with the same scale"

C, W, losses = nmf_fit(X_sub, 4, n_iter=200, seed=SEED)
show("loss after rounds 1, 10, 50, 100 and 200", np.array(losses)[[0, 9, 49, 99, -1]] if len(losses) >= 100 else np.array(losses))
show("length of each synergy (row of W)", np.linalg.norm(W, axis=1), np.ones(4))
assert C.shape == (20000, 4) and W.shape == (4, 16), f"shapes {C.shape}, {W.shape}; expected (20000, 4), (4, 16)"
assert (C >= 0).all() and (W >= 0).all(), "C and W must stay non-negative"
assert len(losses) == 200, f"{len(losses)} losses, expected one per round (200)"
assert all(b <= a * (1 + 1e-9) for a, b in zip(losses, losses[1:])), \
    "the loss went UP at some round: check the order of the updates (W first, then C with the new W)"
assert np.allclose(np.linalg.norm(W, axis=1), 1, atol=1e-6), "each row of W should have length 1 at the end"
r_C, r_W, r_losses = ref.nmf_fit(X_sub, 4, n_iter=200, seed=SEED)
show("final loss", losses[-1], r_losses[-1])
assert abs(losses[-1] / r_losses[-1] - 1) < 0.01, \
    f"final loss {losses[-1]:,.0f} vs reference {r_losses[-1]:,.0f}: record nmf_loss (with the 1/2) after each round"
C_new = nmf_transform(X_sub[:2000], W, n_iter=200, seed=SEED)
assert C_new.shape == (2000, 4) and (C_new >= 0).all()
assert ref.nmf_loss(X_sub[:2000], C_new, W) <= 1.02 * ref.nmf_loss(X_sub[:2000], C[:2000], W), \
    "nmf_transform should rebuild about as well as the fit itself"
same = np.allclose(W, r_W, atol=1e-6)
print(f"✅ NMF works. With 4 synergies the rebuild keeps VAF = {ref.vaf(X_sub, C @ W):.1f}% of the EMG "
      "(Task 2 has you write VAF)" + (" — identical to the reference" if same else ""))

### 📊 The loss falls, and the synergies are all positive
Left: the loss after each round of your NMF, against the best and the worst learning rate of step 1c, all from the
same start. Right: the 4 synergies. **What to look for:** how much faster the multiplicative updates get low with
no tuning at all, and that every synergy weight is ≥ 0. Each synergy tends to sit on a **group of neighbouring
electrodes**, one region of the forearm.

In [ ]:
# TODO: a 14 × 3 figure with 5 panels in a row.
# TODO: panel 1: `losses` from your nmf_fit (label "multiplicative"), the PGD curve from pgd_curves with the LOWEST
#       final loss (label it with its η), and the η = 1e-5 curve (dotted). Log y axis, x = round, legend.
# TODO: panels 2–5: a bar chart of each synergy W[i] over the 16 electrodes, titled "synergy i", x = electrode.

#### Step 1e, part 2 · A test with a known answer: `planted_synergies` and `match_synergies`
On real EMG nobody knows the true synergies, so a good fit can't tell you whether NMF finds the **right** ones.
The standard check is to build fake data where you **planted** the answer, then see whether the method gets it back.
If it fails here, it can't be trusted on real data; if it succeeds, that is evidence (not proof) that it works.

**`planted_synergies`** builds that data. Each synergy is a smooth **bump around the ring** of 16 electrodes (the
electrodes sit around the forearm, so electrode 15 neighbours electrode 0, hence a cosine). The activations are
**sparse** (each synergy is off half the time) and non-negative, plus a little positive noise.

**`match_synergies`** compares the fit with the truth. From Q1a (b), a fit is only defined **up to the scale and
the order** of its synergies, so a raw `W_found == W_true` fails even for a perfect fit. Remove both:
1. **Scale:** compare directions, not lengths. The **cosine similarity** of two rows is $\frac{a \cdot b}{\lVert a\rVert\,\lVert b\rVert}$:
   1 for the same shape, whatever the size.
2. **Order:** try every way of pairing true synergies with fitted ones (each fitted row used once) and keep the
   pairing with the largest **total** cosine. With k = 3 that is only 3! = 6 pairings.

In [ ]:
from itertools import permutations


def planted_synergies(T: int = 2000, seed: int = 0) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Synthetic EMG built from 3 known synergies, so you can check that NMF finds them.

    Recipe (keep this order of random draws so your data matches the reference):
    1. W_true (3, 16): synergy i is a bump around the ring centred on electrode c_i, with c = (2, 7, 12):
       W_true[i, j] = exp(3 * cos(2π (j - c_i) / 16)). Then scale each row to length 1.
    2. rng = np.random.default_rng(seed). C_true (T, 3) = rng.gamma(0.5, 1.0, (T, 3)) times a 0/1 mask that keeps
       each entry with probability 0.5: (rng.random((T, 3)) < 0.5).
    3. X = C_true @ W_true + 0.01 * rng.random((T, 16)).

    Returns
    -------
    X : (T, 16), C_true : (T, 3), W_true : (3, 16), all ≥ 0
    """
    # TODO
    raise NotImplementedError


def match_synergies(W: np.ndarray, W_true: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Pair each true synergy with a different fitted one, ignoring scale and order.

    Parameters
    ----------
    W      : (k, D) fitted synergies, any row lengths, any order
    W_true : (k, D) true synergies

    Returns
    -------
    order : (k,) int array, W[order[i]] is the fitted synergy paired with W_true[i] (each fitted row used once)
    cos   : (k,) float array, the cosine similarity of each pair (1 = same shape)
    The pairing maximises the SUM of the k cosines. Trying every ordering is fine for k ≤ 6.
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on part 2? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

`planted_synergies`
1. Electrode numbers `0 … 15`; for each centre, compute the bump over all 16 electrodes at once; stack the 3 rows.
2. Divide each row by its length.
3. Draw the gamma values, then the mask, then the noise, all from **one** `rng`, in that order.

`match_synergies`
1. Scale every row of both matrices to length 1.
2. One matrix product gives every cosine at once: `S[i, j]` = cosine of true synergy *i* with fitted synergy *j*.
3. Go through every ordering `p` of the fitted rows; its score is `S[0, p[0]] + S[1, p[1]] + …`. Keep the best.
4. Return the best ordering and the k cosines along it.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.linalg.norm(M, axis=1, keepdims=True)` → (k, 1) row lengths; `M / that` scales every row to length 1.
- For unit rows, `Tn @ Wn.T` is the (k, k) table of cosines.
- `permutations(range(3))` yields `(0, 1, 2), (0, 2, 1), (1, 0, 2), …`; `max(items, key=score)` picks the best.
- Fancy indexing picks one entry per row: `S[np.arange(k), order]`.

```python
S = np.array([[0.1, 0.9], [0.8, 0.2]])
max(permutations(range(2)), key=lambda p: S[0, p[0]] + S[1, p[1]])   # (1, 0): 0.9 + 0.8
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def planted_synergies(T=2000, seed=0):
    rng = np.random.default_rng(seed)
    ch = np.arange(16)
    W_true = np.array([np.exp(3 * np.cos(2 * np.pi * (ch - c) / ____)) for c in (2, 7, 12)])
    W_true /= np.linalg.norm(W_true, axis=____, keepdims=True)
    C_true = rng.gamma(0.5, 1.0, (T, 3)) * (rng.random((T, 3)) < ____)
    X = C_true @ W_true + 0.01 * rng.random((T, ____))
    return X, C_true, W_true

def match_synergies(W, W_true):
    Wn = W / np.linalg.norm(W, axis=1, keepdims=True)
    Tn = ____ / np.linalg.norm(____, axis=1, keepdims=True)
    S = Tn @ ____.T
    k = len(W_true)
    best = max(permutations(range(len(W)), k), key=lambda p: sum(S[i, p[i]] for i in range(____)))
    order = np.array(best)
    return order, S[np.arange(k), ____]
```

</details>

In [ ]:
# ✅ Check step 1e, part 2: your test data, your matcher, and does your NMF find the synergies you planted?
X_pl, C_pl, W_pl = planted_synergies(T=2000, seed=SEED)          # X (2000, 16) built from 3 known synergies
r_X, r_C, r_W = ref.planted_synergies(T=2000, seed=SEED)
show("planted synergy 1 (W_true[0])", W_pl[0], r_W[0])
assert X_pl.shape == (2000, 16) and C_pl.shape == (2000, 3) and W_pl.shape == (3, 16), "shapes: X (T, 16), C (T, 3), W (3, 16)"
assert np.allclose(np.linalg.norm(W_pl, axis=1), 1), "scale each planted synergy to length 1"
assert np.allclose(W_pl, r_W), "W_true differs: exp(3·cos(2π(j − c)/16)) for c = 2, 7, 12, then rows of length 1"
assert np.allclose(C_pl, r_C) and np.allclose(X_pl, r_X), "C_true or X differs: one rng; gamma, then the mask, then the noise"

perm, scale = np.array([2, 0, 1]), np.array([[5.0], [0.1], [2.0]])
order_t, cos_t = match_synergies(W_pl[perm] * scale, W_pl)       # a reordered, rescaled copy of the truth
show("matcher on a reordered, rescaled copy: order", order_t, np.argsort(perm))
show("... and the cosine of each pair", cos_t, np.ones(3))
assert np.allclose(cos_t, 1), f"a rescaled, reordered copy should match with cosine 1 (got {np.round(cos_t, 3)}): normalise the rows"
assert np.array_equal(perm[order_t], np.arange(3)), f"order {order_t}, expected {np.argsort(perm)}: order[i] should point at the copy of W_true[i]"
rng_mt = np.random.default_rng(3)
Wa, Wb = rng_mt.random((4, 16)), rng_mt.random((4, 16))
assert np.array_equal(match_synergies(Wa, Wb)[0], ref.match_synergies(Wa, Wb)[0]), "pick the pairing with the largest SUM of cosines"

_, W_found, _ = nmf_fit(X_pl, 3, n_iter=500, seed=SEED)
order, cos = match_synergies(W_found, W_pl)                      # NMF returns synergies in any order: pair them up
assert (cos > 0.99).all(), f"recovered synergies match the planted ones only to cosine {np.round(cos, 3)}"
print(f"✅ NMF recovered all 3 planted synergies (cosine similarity {', '.join(f'{c:.3f}' for c in cos)})")

fig, ax = plt.subplots(1, 3, figsize=(12, 2.6), sharey=True)
for i in range(3):
    ax[i].bar(range(16), W_pl[i], color="0.75", label="planted")
    ax[i].plot(range(16), W_found[order[i]], "o-", color="C2", ms=4, label="found by your NMF")
    ax[i].set_title(f"synergy {i + 1}  (cos = {cos[i]:.3f})"); ax[i].set_xlabel("electrode")
ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()

✍️ **Q1e.** **(a)** Why scale the rows of `W` to length 1 at the end? What would go wrong when comparing synergies from two
fits (or two people) if you didn't? **(b)** At a minimum under the rule W ≥ 0, every entry obeys one of two
patterns (the **KKT conditions**): either W > 0 and its gradient is 0, or W = 0 and its gradient is ≥ 0 (it
"wants" to go negative but isn't allowed). Explain in words why. Then test it: compute
`gC, gW = nmf_grads(X_sub, C, W)` on the fitted model. Compare `np.abs(gW[W > 0.05])` with the size of what the
data asks for, `np.abs(C.T @ X_sub).mean()`. Then look at `gW[W < 1e-3]`. If you find negative gradients there,
what does that say about whether 200 rounds reached the minimum, and about how fast multiplicative updates move an
entry that is near 0?
**(c)** The planted test passed with k = 3. Predict what `nmf_fit(X_pl, 4)` returns, then try it. What does that
tell you about choosing k on real data, where nobody planted anything?

> *Your answer:*

### Step 1f (practice) · A different noise model: KL-divergence NMF
Step 1a's squared error is not arbitrary: it is what you get from a **Gaussian** noise model. Change the noise model
and the loss changes with it. This step derives the loss for a **Poisson** noise model, then has you build the whole
algorithm again (loss → gradient → multiplicative updates → fit) for that loss.

**1. Where the squares came from (Gaussian).** Model each entry as the rebuild plus bell-curve noise:
$X_{tj} \sim \mathcal N(\mu_{tj}, \sigma^2)$ with $\mu_{tj} = (CW)_{tj}$. The density of one entry is
$$p(X_{tj}) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\Big(-\frac{(X_{tj} - \mu_{tj})^2}{2\sigma^2}\Big)$$
The square is the "distance from the centre" inside the bell curve. With independent entries, the probability of
all the data is the product of these; its log is a sum:
$$\log p(X) = \text{const} - \frac{1}{\sigma^2}\cdot\underbrace{\tfrac12 \textstyle\sum_{t,j}(X_{tj} - \mu_{tj})^2}_{L\ \text{from step 1a}}$$
Most likely `C, W` ⇔ largest $\log p$ ⇔ smallest $L$. The hidden assumption: **every entry has the same noise
width σ**, whether the muscle is quiet or loud.

**2. Poisson: noise that grows with the signal.** A Poisson variable with mean $\mu$ has variance $\mu$: big values
are noisier in absolute terms. EMG behaves more like that (a loud channel jitters more than a quiet one). The
Poisson probability is
$$p(x) = \frac{\mu^{x} e^{-\mu}}{x!} \quad\Longrightarrow\quad \log p(x) = x \log \mu - \mu - \log x!$$
Sum over all entries and flip the sign (we minimise). $\log x!$ does not depend on `C, W`, so drop it, and add
$x\log x - x$ (also free of `C, W`) so that a perfect rebuild scores exactly 0. The result is the
**(generalised) KL divergence**:
$$D(X \,\Vert\, CW) = \sum_{t,j} \Big( X_{tj}\log\frac{X_{tj}}{(CW)_{tj}} - X_{tj} + (CW)_{tj} \Big)$$
Each term is ≥ 0 and equals 0 only when $(CW)_{tj} = X_{tj}$ (try $X = 1$ and rebuilds of 0.5, 1, 2).
(The envelope isn't really a count, so "Poisson" is a model of how the noise *scales*, not a literal claim.)

**3. What changes in practice: relative errors.** Compare two misses of the same size, 1:

| entry | rebuild | squared error ½(X − μ)² | KL term |
|---|---|---|---|
| quiet: X = 1 | 2 | 0.5 | 1·log(1/2) − 1 + 2 ≈ **0.31** |
| loud: X = 100 | 101 | 0.5 | 100·log(100/101) − 100 + 101 ≈ **0.005** |

Squared error scores them the same. KL says missing by 1 on a quiet entry is about 60× worse, because there it is a
100% error, and on the loud one only 1%. For small errors, $\text{KL term} \approx \frac{(X - \mu)^2}{2\mu}$: a squared
error **divided by the size of the entry**. So KL is least squares with each entry weighted by 1/size, exactly what
you'd do if the noise variance grows with the signal.

**4. The gradient.** Same chain rule as step 1b. Write $Q = X / (CW)$ (entry by entry, shape T × D). Differentiating
one term by $(CW)_{tj}$ gives $-X_{tj}/(CW)_{tj} + 1 = 1 - Q_{tj}$, so
$$\nabla_W D = C^\top (\mathbf 1 - Q) = \underbrace{C^\top \mathbf 1}_{\text{gives}} - \underbrace{C^\top Q}_{\text{asks}},
\qquad \nabla_C D = (\mathbf 1 - Q)\,W^\top = \mathbf 1 W^\top - Q\,W^\top$$
where $\mathbf 1$ is a T × D matrix of ones. $C^\top \mathbf 1$ is simple: entry $(i, j)$ is $\sum_t C_{ti}$, the
**column sum** of `C` for synergy *i*, the same for every electrode *j*. Likewise $\mathbf 1 W^\top$ has entry
$(t, i) = \sum_j W_{ij}$, the **row sum** of `W`.

**5. The multiplicative updates.** Same trick as step 1d: take a gradient step with a step size per entry,
$H = W / (C^\top \mathbf 1)$, and the minus sign cancels:
$$W \leftarrow W - H \odot \nabla_W D = W \odot \frac{C^\top Q}{C^\top \mathbf 1}, \qquad
C \leftarrow C \odot \frac{Q\,W^\top}{\mathbf 1 W^\top}$$
"Asks over gives" again, and Lee & Seung (2001) proved that this loss never goes up either.

**You write:** `kl_loss`, `kl_grads`, `kl_update_W`, `kl_update_C`, `nmf_fit_kl`. Use `EPS` as the docstrings say, so
nothing divides by 0 or takes `log(0)`.

In [ ]:
def kl_loss(X: np.ndarray, C: np.ndarray, W: np.ndarray) -> float:
    """Generalised KL divergence D(X || C W), summed over every entry.

    Use: sum of  X * log((X + EPS) / (C @ W + EPS)) - X + C @ W   (EPS keeps 0·log 0 = 0 and avoids log 0)

    Parameters
    ----------
    X : (T, D) float ≥ 0, C : (T, k) float ≥ 0, W : (k, D) float ≥ 0

    Returns
    -------
    float ≥ 0. 0 means C @ W rebuilds X exactly.
    """
    # TODO
    raise NotImplementedError


def kl_grads(X: np.ndarray, C: np.ndarray, W: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Gradients of kl_loss with respect to C and W.

    With G = 1 - X / (C @ W + EPS)   (T, D):
    gC : (T, k) = G W.T
    gW : (k, D) = C.T G
    """
    # TODO
    raise NotImplementedError


def kl_update_W(X: np.ndarray, C: np.ndarray, W: np.ndarray) -> np.ndarray:
    """Multiplicative KL update of the synergies: W ⊙ (Cᵀ Q) / (Cᵀ 1 + EPS), with Q = X / (C @ W + EPS).

    Cᵀ 1 has entry (i, j) = column sum of C for synergy i (the same for every electrode j).

    Returns
    -------
    W_new : (k, D) float ≥ 0 (a new array)
    """
    # TODO
    raise NotImplementedError


def kl_update_C(X: np.ndarray, C: np.ndarray, W: np.ndarray) -> np.ndarray:
    """Multiplicative KL update of the activations: C ⊙ (Q Wᵀ) / (1 Wᵀ + EPS), with Q = X / (C @ W + EPS).

    1 Wᵀ has entry (t, i) = row sum of W for synergy i (the same for every frame t).

    Returns
    -------
    C_new : (T, k) float ≥ 0 (a new array)
    """
    # TODO
    raise NotImplementedError


def nmf_fit_kl(X: np.ndarray, k: int, n_iter: int = 300, seed: int = 0
               ) -> tuple[np.ndarray, np.ndarray, list[float]]:
    """Like your nmf_fit, but with the KL updates.

    Same start (your nmf_init), same order each round (kl_update_W, then kl_update_C with the new W), record
    kl_loss after each round, and at the end scale each row of W to length 1 (C scaled up to compensate).

    Returns
    -------
    C : (T, k), W : (k, D) with rows of length 1, losses : list of n_iter floats (kl_loss after each round)
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on step 1f? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `kl_loss`: rebuild `R = C @ W` once, then add up the formula over every entry. Return a plain float.
2. `kl_grads`: compute `G = 1 - X / (R + EPS)` once. Its shape is (T, D), like the residual in step 1b; then the
   same two products as `nmf_grads`, with `G` in place of `C W - X`.
3. `kl_update_W`: compute the ratio `Q`; the top is `C.T @ Q` (k, D); the bottom is the column sums of `C`, one number
   per synergy, which must be **stretched across all D electrodes**.
4. `kl_update_C`: the top is `Q @ W.T` (T, k); the bottom is the row sums of `W`, one per synergy, stretched across all T frames.
5. `nmf_fit_kl`: copy your `nmf_fit` and swap in the KL pieces.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.log` works entry by entry. `C.sum(axis=0)` sums down the rows: shape (k,). `W.sum(axis=1)` sums along each row: shape (k,).
- **Broadcasting:** a (k, 1) column divides every column of a (k, D) matrix; a (1, k) row divides every row of a
  (T, k) matrix. `v[:, None]` turns shape (k,) into (k, 1); `v[None, :]` turns it into (1, k).

```python
C = np.array([[1., 2.], [3., 4.], [5., 6.]])   # (T=3, k=2)
C.sum(axis=0)                                  # array([ 9., 12.])   one per synergy
C.sum(axis=0)[:, None].shape                   # (2, 1): divides a (2, D) top row by row
```

- Check the "gives" claim yourself: `C.T @ np.ones((3, 4))` has every column equal to `C.sum(axis=0)`.

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def kl_loss(X, C, W):
    R = C @ W
    return float(np.sum(X * np.log((X + EPS) / (____ + EPS)) - X + ____))

def kl_grads(X, C, W):
    G = 1.0 - X / (C @ W + ____)
    return G @ ____.T, ____.T @ G

def kl_update_W(X, C, W):
    Q = X / (C @ W + EPS)
    return W * (C.T @ ____) / (C.sum(axis=____)[:, None] + EPS)

def kl_update_C(X, C, W):
    Q = X / (C @ W + EPS)
    return C * (Q @ ____) / (W.sum(axis=____)[None, :] + EPS)

def nmf_fit_kl(X, k, n_iter=300, seed=0):
    C, W = nmf_init(X, k, seed)
    losses = []
    for _ in range(n_iter):
        W = kl_update_W(X, ____, W)
        C = kl_update_C(X, C, ____)
        losses.append(kl_loss(X, C, W))
    norms = np.linalg.norm(W, axis=1) + EPS
    return C * norms, W / norms[:, None], losses
```

</details>

In [ ]:
# ✅ Check step 1f
# the loss: zero for a perfect rebuild, never negative, and the table's two worked examples
assert isinstance(kl_loss(X_t, C_t, W_t), float), "return a plain float: wrap the result in float(...)"
assert abs(kl_loss(C_t @ W_t, C_t, W_t)) < 1e-8, "if X is exactly C @ W the loss should be 0"
assert np.isclose(kl_loss(X_t, C_t, W_t), ref.kl_loss(X_t, C_t, W_t)), \
    f"loss {kl_loss(X_t, C_t, W_t):.6f}, expected {ref.kl_loss(X_t, C_t, W_t):.6f}: X·log(X/CW) − X + CW, summed"
one = np.ones((1, 1))
show("KL loss on the tiny problem", kl_loss(X_t, C_t, W_t), ref.kl_loss(X_t, C_t, W_t))
show("the table's examples: X = 1 rebuilt as 2, X = 100 rebuilt as 101",
     np.array([kl_loss(one, one, 2 * one), kl_loss(100 * one, one, 101 * one)]),
     np.array([np.log(0.5) + 1, 100 * np.log(100 / 101) + 1]))
assert np.isclose(kl_loss(one, one, 2 * one), np.log(0.5) + 1, atol=1e-6), "X = 1, rebuild 2 should cost log(1/2) + 1 ≈ 0.31"
assert np.isclose(kl_loss(100 * one, one, 101 * one), 100 * np.log(100 / 101) + 1, atol=1e-6), "X = 100, rebuild 101 should cost ≈ 0.005"

# the gradient, tested with YOUR numerical_grad
gC, gW = kl_grads(X_t, C_t, W_t)
assert gC.shape == C_t.shape and gW.shape == W_t.shape, f"shapes {gC.shape}, {gW.shape}; expected {C_t.shape}, {W_t.shape}"
num_gW = numerical_grad(lambda W_: ref.kl_loss(X_t, C_t, W_), W_t)
num_gC = numerical_grad(lambda C_: ref.kl_loss(X_t, C_, W_t), C_t)
show("dD/dW: your formula vs the numerical slope", gW, num_gW)
show("dD/dC: your formula vs the numerical slope", gC, num_gC)
assert np.allclose(gW, num_gW, atol=1e-5), f"dD/dW is off by up to {np.abs(gW - num_gW).max():.2e}: C.T @ (1 − X / CW)"
assert np.allclose(gC, num_gC, atol=1e-5), f"dD/dC is off by up to {np.abs(gC - num_gC).max():.2e}: (1 − X / CW) @ W.T"

# the updates: match the reference, are gradient steps, are symmetric, and never raise the loss
C_keep, W_keep = C_t.copy(), W_t.copy()
W1, C1 = kl_update_W(X_t, C_t, W_t), kl_update_C(X_t, C_t, W_t)
show("W after one kl_update_W", W1, ref.kl_update_W(X_t, C_t, W_t))
assert np.array_equal(C_t, C_keep) and np.array_equal(W_t, W_keep), "an update changed its input: return a new array"
assert np.allclose(W1, ref.kl_update_W(X_t, C_t, W_t)), "kl_update_W differs: W * (C.T @ Q) / (column sums of C, as a (k, 1) column)"
assert np.allclose(C1, ref.kl_update_C(X_t, C_t, W_t)), "kl_update_C differs: C * (Q @ W.T) / (row sums of W, as a (1, k) row)"
assert np.allclose(W1, W_t - W_t / C_t.sum(axis=0)[:, None] * ref.kl_grads(X_t, C_t, W_t)[1], atol=1e-8)   # a gradient step
assert np.allclose(C1, kl_update_W(X_t.T, W_t.T, C_t.T).T), "kl_update_C should equal kl_update_W(X.T, W.T, C.T).T"
rng_k = np.random.default_rng(1)
for _ in range(500):
    Xr, Cr, Wr = rng_k.random((8, 6)), rng_k.random((8, 3)), rng_k.random((3, 6))
    Wn = kl_update_W(Xr, Cr, Wr); Cn = kl_update_C(Xr, Cr, Wn)
    assert (Wn >= 0).all() and (Cn >= 0).all()
    assert ref.kl_loss(Xr, Cr, Wn) <= ref.kl_loss(Xr, Cr, Wr) + 1e-12, "kl_update_W raised the loss"
    assert ref.kl_loss(Xr, Cn, Wn) <= ref.kl_loss(Xr, Cr, Wn) + 1e-12, "kl_update_C raised the loss"

# the whole fit: the loss never rises, rows have length 1, and it finds the planted synergies
C_kp, W_kp, L_kp = nmf_fit_kl(X_pl, 3, n_iter=500, seed=SEED)
show("KL loss after rounds 1, 10, 100 and 500 (planted data)", np.array(L_kp)[[0, 9, 99, -1]] if len(L_kp) >= 100 else np.array(L_kp))
assert len(L_kp) == 500 and all(b <= a * (1 + 1e-9) for a, b in zip(L_kp, L_kp[1:])), \
    "record kl_loss once per round, and it should never go up: kl_update_W first, then kl_update_C with the new W"
assert np.allclose(np.linalg.norm(W_kp, axis=1), 1, atol=1e-6), "scale each row of W to length 1 at the end"
_, cos_kp = match_synergies(W_kp, W_pl)
assert (cos_kp > 0.99).all(), f"KL-NMF matches the planted synergies only to cosine {np.round(cos_kp, 3)}"
print(f"✅ KL-NMF works: gradients match the numerical slopes, the loss never rises, and it recovers the planted "
      f"synergies (cosine {', '.join(f'{c:.3f}' for c in cos_kp)})")

### 📊 Experiment: squared error vs KL on your EMG
Fit both versions to the same 20,000 training frames (`X_sub`), with k = 4, 300 rounds and the same seed, then
compare them three ways:
1. **A 2 × 2 table of scores:** each fit scored by **both** losses (`nmf_loss` and `kl_loss`). Store it as
   `loss_table = {"squared": (nmf_loss, kl_loss), "kl": (nmf_loss, kl_loss)}`, keyed by the fit.
2. **Do they find the same synergies?** `match_synergies(W_kl, W_sq)`: the cosine of each pair. Plot the pairs
   overlaid, one bar chart per synergy (squared-error fit as bars, KL fit as dots on a line).
3. **Where do they differ?** The **relative error** `|rebuild − X| / X` per entry. Report its median over the
   *quiet* entries (X below its 25th percentile) and over the *loud* ones (above its 75th percentile), for both fits.
   Store it as `rel_err = {"squared": (quiet, loud), "kl": (quiet, loud)}`.

**What to look for:** who wins each column of the table, how alike the synergies are, and whether the gain from KL
sits in the quiet or the loud entries.

In [ ]:
C_sq, W_sq, _ = None, None, None      # TODO: your nmf_fit on X_sub, k = 4, n_iter = 300, seed = SEED
C_kl, W_kl, _ = None, None, None      # TODO: your nmf_fit_kl, same settings
loss_table: dict[str, tuple[float, float]] = {}   # TODO: fit → (nmf_loss, kl_loss), for "squared" and "kl"
rel_err: dict[str, tuple[float, float]] = {}      # TODO: fit → (median relative error on quiet, on loud entries)
# TODO: print loss_table as a small table; print the cosines from match_synergies(W_kl, W_sq); print rel_err
# TODO: one row of 4 bar charts: W_sq[i] as bars, the matched W_kl row as dots on a line, cosine in each title

#### 💡 Stuck on the experiment? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Two fits, same data and settings. Rebuild each once: `C @ W`.
2. The table: score each rebuild with both losses; four numbers.
3. Matching: `order, cos = match_synergies(W_kl, W_sq)` pairs `W_kl[order[i]]` with `W_sq[i]`.
4. Relative error: one boolean mask for quiet entries and one for loud ones, both from `X_sub`'s own percentiles;
   take the median of `|R − X| / X` inside each mask.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.quantile(X_sub, 0.25)` is one number for the whole matrix; `X_sub < that` is a (T, 16) mask.
- `A[mask]` pulls out the masked entries as a flat array; `np.median` of that is one number.

```python
X = np.array([[1., 10.], [2., 20.]]); R = X + 1
q = X < np.quantile(X, 0.25)                 # [[True, False], [False, False]]
np.median(np.abs(R[q] - X[q]) / X[q])        # 1.0: missing 1 on an entry of 1 is a 100% error
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
C_sq, W_sq, _ = nmf_fit(X_sub, 4, n_iter=300, seed=SEED)
C_kl, W_kl, _ = nmf_fit_kl(____, 4, n_iter=300, seed=SEED)
quiet, loud = X_sub < np.quantile(X_sub, 0.25), X_sub > np.quantile(X_sub, ____)
for name, (Cf, Wf) in {"squared": (C_sq, W_sq), "kl": (C_kl, W_kl)}.items():
    R = Cf @ Wf
    loss_table[name] = (nmf_loss(X_sub, Cf, Wf), kl_loss(____, Cf, Wf))
    rel = np.abs(R - X_sub) / X_sub
    rel_err[name] = (float(np.median(rel[____])), float(np.median(rel[loud])))
order, cos = match_synergies(W_kl, ____)
```

</details>

In [ ]:
# ✅ Check the experiment
assert set(loss_table) == {"squared", "kl"} and set(rel_err) == {"squared", "kl"}, 'keys: "squared" and "kl"'
for name in ("squared", "kl"):
    show(f'fit "{name}": squared loss, KL loss, quiet rel. error, loud rel. error', np.array([*loss_table[name], *rel_err[name]]))
assert np.isclose(loss_table["squared"][0], ref.nmf_loss(X_sub, C_sq, W_sq)) and \
       np.isclose(loss_table["kl"][1], ref.kl_loss(X_sub, C_kl, W_kl)), "each entry is (nmf_loss, kl_loss) of that fit"
assert loss_table["squared"][0] < loss_table["kl"][0], "the squared-error fit should have the lower squared error"
assert loss_table["kl"][1] < loss_table["squared"][1], "the KL fit should have the lower KL loss"
assert rel_err["kl"][0] < rel_err["squared"][0], "KL should have the smaller relative error on the quiet entries"
print("✅ Each fit wins on the score it was trained for, and KL's gain sits in the quiet entries")

✍️ **Q1f.** **(a)** Each fit wins its own column of the table. Why must that be (nearly) guaranteed? Task 2's VAF is
built from squared error: which method does reporting VAF quietly favour, and what does that mean when a paper
compares methods using one score? **(b)** Use the approximation $\text{KL term} \approx (X - \mu)^2 / 2\mu$ to explain why
KL's gain is on the quiet entries and not the loud ones. **(c)** Which noise model fits EMG envelopes better, and what
plot of the residuals would tell you? (Idea: bin the entries by the size of their rebuild and plot the variance of
the residual in each bin. Flat → Gaussian; growing in a straight line → Poisson-like. Try it if you like.)
**(d)** The two fits' synergies have cosines of roughly 0.9 to 1. Does the choice of loss change the stage 1 claim
"this person's EMG has about k synergies"? What would?

> *Your answer:*

---
## Task 2 · VAF: how much of the EMG is explained?
Step 1a's loss is a raw number that depends on how loud the data is. VAF (*variance accounted for*) turns it
into a percentage and is the standard score in the synergy literature:

$$\text{VAF} = 100 \times \left(1 - \frac{\sum (X - \hat X)^2}{\sum X^2}\right)$$

The sums run over **every** entry (all frames and all channels). 100 = perfect rebuild; 0 = no better than
rebuilding everything as 0. In step 1a's terms, VAF = 100 × (1 − L(C, W) / L(0)).

It is **uncentered**: the denominator is `X` itself, not `X` minus its mean. That matters for NMF, which has
no "mean" term: rebuilding the average activity level is part of its job, so it should get credit for it.

In [ ]:
def vaf(X: np.ndarray, X_hat: np.ndarray) -> float:
    """Variance accounted for, in %.

    Parameters
    ----------
    X     : (T, D) float, the real data (e.g. EMG, non-negative)
    X_hat : (T, D) float, the rebuilt data (same shape as X)

    Returns
    -------
    float, at most 100. 100 = perfect rebuild. Uncentered: compare the error against X, not X minus its mean.

    Check: vaf(X, X) → 100.0 and vaf(X, np.zeros_like(X)) → 0.0
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 2? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. The error at every entry is `X - X_hat`. Square it, then add up **all** of the squares into one number.
2. Do the same for `X` itself: square every entry and add them all up.
3. Divide the first by the second, subtract from 1, multiply by 100.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `a ** 2` squares every entry. `np.sum(a)` with **no** `axis=` adds up every entry into one number.
- Wrap the result in `float(...)` so you return a plain number, not a NumPy scalar.

```python
a = np.array([[1.0, 2.0], [3.0, 0.0]])
np.sum(a ** 2)        # 14.0  (1 + 4 + 9 + 0)
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def vaf(X: np.ndarray, X_hat: np.ndarray) -> float:
    err = np.sum((____ - ____) ** 2)
    tot = np.sum(____ ** 2)
    return float(100 * (1 - ____ / ____))
```

</details>

In [ ]:
# ✅ Check Task 2
X_toy = np.random.default_rng(0).random((50, 16))
show("vaf(X, X): should be 100", vaf(X_toy, X_toy), 100.0)
show("vaf(X, zeros): should be 0", vaf(X_toy, np.zeros_like(X_toy)), 0.0)
show("vaf(X, X + 0.1)", vaf(X_toy, X_toy + 0.1), ref.vaf(X_toy, X_toy + 0.1))
assert abs(vaf(X_toy, X_toy) - 100) < 1e-9, f"vaf(X, X) = {vaf(X_toy, X_toy)}, expected 100"
assert abs(vaf(X_toy, np.zeros_like(X_toy))) < 1e-9, "vaf(X, zeros) should be 0: did you centre X?"
noisy = X_toy + 0.1
assert abs(vaf(X_toy, noisy) - ref.vaf(X_toy, noisy)) < 1e-9, \
    f"vaf = {vaf(X_toy, noisy):.4f}, expected {ref.vaf(X_toy, noisy):.4f}"
print(f"✅ VAF works. Adding 0.1 to every entry gives VAF = {vaf(X_toy, noisy):.1f}%")

✍️ **Q2.** VAF is *uncentered*. Compute the VAF you get by rebuilding every **test** frame with nothing but the
**training mean** of each channel (`np.tile(X_train.mean(axis=0), (len(X_test), 1))`, after Task 4 defines them,
or on `X_toy` now). What does that number tell you about how to read "90% VAF"?

> *Your answer:*

---
## Task 3 · PCA from scratch, with the SVD
PCA finds the directions along which the data varies most. With the **singular value decomposition (SVD)** it
takes three lines:

1. **Centre:** subtract each column's mean, `Xc = X - mean`. PCA describes variation *around* the average.
2. **SVD:** `U, S, Vt = np.linalg.svd(Xc, full_matrices=False)` splits `Xc = U · diag(S) · Vt`.
   The **rows of `Vt`** are the principal directions (each of length 1, each at right angles to the others),
   already sorted from most to least variance. `S**2` is proportional to the variance along each one.
3. **Keep the top k:** `comps = Vt[:k]`, shape (k, D).

Then two small functions use them:
- **transform** (data → scores): centre, then project onto each direction: `(X - mean) @ comps.T` → (T, k)
- **inverse** (scores → rebuilt data): add up the directions weighted by the scores, then add the mean back:
  `scores @ comps + mean` → (T, D)

**One detail: signs.** An SVD direction and its negative are equally valid, so different computers can flip
them. To make answers comparable, flip each component so its **largest-magnitude entry is positive**.

In [ ]:
def pca_fit(X: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray]:
    """Fit PCA with k components using the SVD.

    Parameters
    ----------
    X : (T, D) float, data (rows = frames)
    k : int, number of components to keep

    Returns
    -------
    mean  : (D,) float, the mean of each column
    comps : (k, D) float, the top-k principal directions (rows of length 1), most variance first,
            each flipped so that its largest-magnitude entry is positive
    """
    # TODO
    raise NotImplementedError


def pca_transform(X: np.ndarray, mean: np.ndarray, comps: np.ndarray) -> np.ndarray:
    """(T, D) data → (T, k) scores: centre with mean, then project onto comps."""
    # TODO
    raise NotImplementedError


def pca_inverse(scores: np.ndarray, mean: np.ndarray, comps: np.ndarray) -> np.ndarray:
    """(T, k) scores → (T, D) rebuilt data: weighted sum of comps, plus the mean."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 3? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `pca_fit`: compute the column means, centre `X`, run the SVD, keep the first `k` rows of `Vt`.
2. Fix the signs: for each kept row, find the entry with the largest absolute value. If that entry is negative,
   multiply the whole row by −1.
3. `pca_transform` and `pca_inverse` are one line each: the two formulas in the text above.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `X.mean(axis=0)` → one mean per column, shape (D,). `X - mean` subtracts it from every row (broadcasting).
- `np.linalg.svd(Xc, full_matrices=False)` returns `U` (T, D), `S` (D,), `Vt` (D, D).
- `np.abs(c).argmax(axis=1)` → for each row, the column of its largest-magnitude entry.
- `np.sign(v)` → +1 or −1. Multiplying `comps * signs[:, None]` flips whole rows.

```python
c = np.array([[0.2, -0.9], [0.6, 0.1]])
big = np.abs(c).argmax(axis=1)          # array([1, 0])
signs = np.sign(c[np.arange(2), big])   # array([-1.,  1.])
c * signs[:, None]                      # [[-0.2, 0.9], [0.6, 0.1]]: row 0 flipped
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def pca_fit(X: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray]:
    mean = X.mean(axis=____)
    _, _, Vt = np.linalg.svd(X - ____, full_matrices=False)
    comps = Vt[:____]
    big = np.abs(comps).argmax(axis=1)
    signs = np.sign(comps[np.arange(k), ____])
    return mean, comps * signs[:, None]


def pca_transform(X, mean, comps):
    return (X - ____) @ ____.T


def pca_inverse(scores, mean, comps):
    return scores @ ____ + ____
```

</details>

In [ ]:
# ✅ Check Task 3
X_sub = emg[train][:5000]
mean, comps = pca_fit(X_sub, 4)
r_mean, r_comps = ref.pca_fit(X_sub, 4)
show("mean of each electrode", mean, r_mean)
show("first component (PC 1)", comps[0], r_comps[0])
show("length of each component", np.linalg.norm(comps, axis=1), np.ones(4))
assert mean.shape == (16,) and comps.shape == (4, 16), f"shapes {mean.shape}, {comps.shape}; expected (16,), (4, 16)"
assert np.allclose(np.linalg.norm(comps, axis=1), 1), "each component should have length 1"
assert np.allclose(mean, r_mean), "mean differs: did you average down the rows (axis=0)?"
assert np.allclose(comps, r_comps, atol=1e-6), "components differ: check the centring, the [:k] slice and the signs"
scores = pca_transform(X_sub, mean, comps)
assert scores.shape == (5000, 4), f"scores shape {scores.shape}, expected (5000, 4)"
X_back = pca_inverse(scores, mean, comps)
assert np.allclose(X_back, ref.pca_inverse(ref.pca_transform(X_sub, r_mean, r_comps), r_mean, r_comps)), "rebuild differs"
assert np.allclose(pca_inverse(pca_transform(X_sub, *pca_fit(X_sub, 16)), *pca_fit(X_sub, 16)), X_sub), \
    "with all 16 components the rebuild should be perfect"
print(f"✅ PCA works. 4 components explain VAF = {vaf(X_sub, X_back):.1f}% of these frames")

### 📊 What the PCA components look like
Each bar chart is one component: a weight per electrode. Notice the **negative** weights. PCA describes
variation as "more of this electrode *and less of that one*", but a muscle can't be *negatively* active.
Compare them with your NMF synergies from Task 1: every NMF weight is ≥ 0.

In [ ]:
# TODO: one row of 4 bar charts sharing the y axis, one per PCA component comps[i] over the 16 electrodes.
#       Colour positive weights blue and negative ones red, draw a thin black line at 0, title each "PC i",
#       label the x axis "electrode" and the first y axis "weight".
#       (Tool: ax.bar accepts a list of colours, one per bar, e.g. np.where(comps[i] >= 0, "C0", "C3").)

✍️ **Q3.** PCA components have negative weights. What would a negative weight on an electrode mean
*physically*, if a component were a muscle synergy? Why is that a problem for calling PCA components "synergies"?

> *Your answer:*

---
## Task 4 · NMF and PCA for k = 1 … 10, scored on test data
Now use your own functions on the full data. For each k:
- **NMF:** `nmf_fit` on `X_train`, then `nmf_transform` on `X_test` with that `W`, then VAF on `X_test`.
- **PCA:** `pca_fit` on `X_train`, then `pca_inverse(pca_transform(X_test, …), …)`, then VAF on `X_test`.

Fitting on train and scoring on test asks whether the synergies **generalise** to later data, not just whether
they can memorise the training frames. This takes about a minute (NMF is the slow part).

In [ ]:
ks = list(range(1, 11))
vaf_nmf: list[float] = []      # length 10 when done: test VAF (%) for k = 1 … 10
vaf_pca: list[float] = []
nmf_W: dict[int, np.ndarray] = {}                       # k → W (k, 16)            (Task 8 reuses it)
pca_fits: dict[int, tuple[np.ndarray, np.ndarray]] = {} # k → (mean, comps)

X_train = emg[train]           # (T_train, 16)
X_test  = emg[test]            # (T_test, 16)

for k in ks:
    # TODO: NMF: fit on X_train (seed=SEED), store W in nmf_W[k], rebuild X_test with nmf_transform, append the VAF
    # TODO: PCA: fit on X_train, store (mean, comps) in pca_fits[k], rebuild X_test, append the VAF
    ...
    print(f"k = {k:2d}   NMF {vaf_nmf[-1]:5.1f}%   PCA {vaf_pca[-1]:5.1f}%")

#### 💡 Stuck on Task 4? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

Inside the loop, two blocks of three lines each:
1. **NMF:** fit → keep `W` → find test activations with `W` fixed → rebuild as `C_test @ W` → VAF against `X_test`.
2. **PCA:** fit → keep `(mean, comps)` → scores of `X_test` → rebuild → VAF against `X_test`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `_` is the usual name for a returned value you don't need: `_, W, _ = nmf_fit(...)`.
- `pca_fit` returns a tuple, so you can unpack it straight back in: `pca_transform(X, *pca_fits[k])`.

```python
fit = (np.zeros(2), np.eye(2))     # (mean, comps)
mean, comps = fit                  # unpack
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
for k in ks:
    _, W, _ = nmf_fit(____, k, seed=SEED)
    nmf_W[k] = W
    C_test = nmf_transform(____, W, seed=SEED)
    vaf_nmf.append(vaf(X_test, ____ @ ____))

    mean, comps = pca_fit(____, k)
    pca_fits[k] = (mean, comps)
    X_hat = pca_inverse(pca_transform(____, mean, comps), mean, comps)
    vaf_pca.append(vaf(____, ____))
    print(f"k = {k:2d}   NMF {vaf_nmf[-1]:5.1f}%   PCA {vaf_pca[-1]:5.1f}%")
```

</details>

In [ ]:
# ✅ Check Task 4 (about 10 s)
assert len(vaf_nmf) == len(vaf_pca) == 10, "one VAF per k = 1 … 10 for each method"
r_pca = [vaf(X_test, ref.pca_inverse(ref.pca_transform(X_test, *ref.pca_fit(X_train, k)), *ref.pca_fit(X_train, k)))
         for k in ks]
show("PCA test VAF (%) for k = 1 … 10", np.array(vaf_pca), np.array(r_pca))
show("NMF test VAF (%) for k = 1 … 10", np.array(vaf_nmf))
assert np.allclose(vaf_pca, r_pca, atol=1e-6), "PCA VAFs differ from the reference: fit on X_train, score on X_test"
_, r_W4, _ = ref.nmf_fit(X_train, 4, seed=SEED)
r_nmf4 = vaf(X_test, ref.nmf_transform(X_test, r_W4, seed=SEED) @ r_W4)
show("NMF test VAF (%) at k = 4", vaf_nmf[3], r_nmf4)
assert abs(vaf_nmf[3] - r_nmf4) < 0.5, f"NMF VAF at k = 4 is {vaf_nmf[3]:.2f}%, reference {r_nmf4:.2f}%"
print("✅ VAF curves match the reference")

✍️ **Q4.** (a) The PCA curve is at or above the NMF curve at every k. Why must that be true (think about what
PCA is optimal for, and what NMF is not allowed to do)? (b) On the test set we **refit `C`** freely and only
keep `W` fixed. Is that a strict test of the synergies? Try it: make a *random* non-negative `W` (4 × 16, rows of
length 1) and score it on `X_test` with `nmf_transform`. How far above it is the real NMF?

> *Your answer:*

---
## Task 5 · Fairer reference points: centred VAF and random synergies
Two things make the Task 4 numbers look better than they are (Q2 and Q4 hinted at both):
1. **Uncentered VAF gives credit for the mean.** **Centred VAF** compares the error with the data *minus its
   column means*, so only the moment-to-moment variation counts:
   $$\text{VAF}_c = 100 \times \left(1 - \frac{\sum (X - \hat X)^2}{\sum (X - \bar X)^2}\right)$$
2. **`C` is refit on the test frames.** Any non-negative `W` gets some credit for free. The honest reference is
   a **null**: random non-negative synergies of the same size, with `C` refit in exactly the same way. Real
   synergies only mean something by **how far they beat this null**.

This task adds three curves to Task 4's: centred VAF for NMF and PCA, and (uncentred) VAF for random `W`.
It reuses `nmf_W` and `pca_fits`, so nothing is refit (about 30 s).

In [ ]:
def vaf_centred(X: np.ndarray, X_hat: np.ndarray) -> float:
    """Centred variance accounted for, in %.

    Parameters
    ----------
    X, X_hat : (T, D) float, real and rebuilt data

    Returns
    -------
    float, at most 100. Like vaf, but the denominator is the sum of squares of X minus its column means.
    Check: vaf_centred(X, np.tile(X.mean(axis=0), (len(X), 1))) → 0.0
    """
    # TODO
    raise NotImplementedError


def random_W(k: int, D: int, seed: int) -> np.ndarray:
    """k random non-negative synergies (k, D), each row scaled to length 1.

    Use rng = np.random.default_rng(seed) and rng.random((k, D)), so your answer matches the reference.
    """
    # TODO
    raise NotImplementedError


vaf_nmf_c: list[float] = []    # length 10: centred test VAF of NMF for k = 1 … 10
vaf_pca_c: list[float] = []    # length 10: centred test VAF of PCA
vaf_rand: list[float] = []     # length 10: (uncentred) test VAF of random_W(k, 16, SEED), C refit with nmf_transform
vaf_mean_only = None           # float: VAF of rebuilding every test frame with the training mean (Q1)
for k in ks:
    # TODO: NMF rebuild of X_test with nmf_W[k] → centred VAF; PCA rebuild with pca_fits[k] → centred VAF;
    # TODO: random W → nmf_transform → VAF
    ...

#### 💡 Stuck on Task 5? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `vaf_centred` is your `vaf` with one change: subtract the column means from `X` in the denominator.
2. `random_W`: draw a (k, D) array of uniform numbers in [0, 1), then divide each row by its length.
3. In the loop, rebuild `X_test` three ways (NMF with `nmf_W[k]`, PCA with `pca_fits[k]`, random `W` with
   `nmf_transform`) and append the scores.
4. `vaf_mean_only`: `np.tile(X_train.mean(axis=0), (len(X_test), 1))` is the "rebuild".

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `X.mean(axis=0)` → (D,) column means; `X - X.mean(axis=0)` centres every column.
- `W / np.linalg.norm(W, axis=1, keepdims=True)` scales each row to length 1.

```python
X = np.array([[1.0, 2.0], [3.0, 4.0]])
X - X.mean(axis=0)          # [[-1., -1.], [ 1.,  1.]]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def vaf_centred(X, X_hat):
    return float(100 * (1 - np.sum((X - X_hat) ** 2) / np.sum((X - X.mean(axis=____)) ** 2)))

def random_W(k, D, seed):
    W = np.random.default_rng(seed).random((____, ____))
    return W / np.linalg.norm(W, axis=1, keepdims=____)

for k in ks:
    W = nmf_W[k]
    vaf_nmf_c.append(vaf_centred(X_test, nmf_transform(X_test, W, seed=SEED) @ ____))
    vaf_pca_c.append(vaf_centred(X_test, pca_inverse(pca_transform(X_test, *pca_fits[k]), *pca_fits[____])))
    Wr = random_W(k, 16, SEED)
    vaf_rand.append(vaf(X_test, nmf_transform(X_test, ____, seed=SEED) @ Wr))
vaf_mean_only = vaf(X_test, np.tile(X_train.mean(axis=0), (len(X_test), 1)))
```

</details>

In [ ]:
# ✅ Check Task 5
X_toy = np.random.default_rng(0).random((50, 16))
show("centred VAF of the column means alone: should be 0", vaf_centred(X_toy, np.tile(X_toy.mean(axis=0), (50, 1))), 0.0)
show("random_W(4, 16, SEED): first synergy", random_W(4, 16, SEED)[0], ref.random_W(4, 16, SEED)[0])
show("centred test VAF (%) for k = 1 … 10, NMF", np.array(vaf_nmf_c))
show("centred test VAF (%) for k = 1 … 10, PCA", np.array(vaf_pca_c))
show("uncentred test VAF (%) of random W for k = 1 … 10", np.array(vaf_rand))
show("VAF (%) of the training mean alone", vaf_mean_only)
assert abs(vaf_centred(X_toy, np.tile(X_toy.mean(axis=0), (50, 1)))) < 1e-9, "the mean alone should score 0 centred VAF"
assert abs(vaf_centred(X_toy, X_toy + 0.05) - ref.vaf_centred(X_toy, X_toy + 0.05)) < 1e-9
assert np.allclose(random_W(4, 16, SEED), ref.random_W(4, 16, SEED)), "use np.random.default_rng(seed).random((k, D))"
assert len(vaf_nmf_c) == len(vaf_pca_c) == len(vaf_rand) == 10
assert abs(vaf_pca_c[3] - ref.vaf_centred(X_test, ref.pca_inverse(ref.pca_transform(X_test, *pca_fits[4]), *pca_fits[4]))) < 1e-6
Wr4 = ref.random_W(4, 16, SEED)
assert abs(vaf_rand[3] - ref.vaf(X_test, ref.nmf_transform(X_test, Wr4, seed=SEED) @ Wr4)) < 0.5
print(f"✅ At k = 4: NMF {vaf_nmf[3]:.1f}% uncentred / {vaf_nmf_c[3]:.1f}% centred; random W {vaf_rand[3]:.1f}%; "
      f"mean only {vaf_mean_only:.1f}%")

✍️ **Q5.** At k = 4, how many points does NMF beat random synergies by, and how many points does it beat the mean-only
rebuild by? Which of the three numbers (VAF, centred VAF, gap over random) would you put in a paper, and why?

> *Your answer:*

---
## Task 6 · Plot VAF against k and choose k
Two panels. **Left:** uncentred VAF (%) against k for NMF, PCA and **random synergies** (the null, dashed),
with a dotted line at the 90% threshold. **Right:** centred VAF for NMF and PCA. Choose k where the NMF curve
**flattens** (adding a synergy barely helps any more), and look at where it first passes 90%. Say which rule you
used: they may disagree, and that's worth discussing with the team. Task 7 then checks your choice on held-out stages.

In [ ]:
# TODO: left panel: vaf_nmf, vaf_pca and vaf_rand (dashed) against ks, with a dotted line at 90%
# TODO: right panel: vaf_nmf_c and vaf_pca_c against ks
# TODO: choose k; write it here and in the Result cell at the bottom

K_CHOSEN: int | None = None    # e.g. 4

#### 💡 Stuck on Task 6? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Make one axis, draw two lines (NMF, PCA) against `ks`, add a horizontal dashed line at 90.
2. Label the axes, add a legend.
3. Read where the NMF curve flattens and where it crosses 90%. A helpful extra: plot the **gain** from each extra
   synergy, `np.diff(vaf_nmf)`, and see where it drops below about 1–2 points.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `ax.plot(x, y, marker="o", label="NMF")` draws a line with dots. `ax.axhline(90, ls="--", color="gray")`.
- `ax.legend()` shows the labels.

```python
np.diff([77, 85, 89, 92, 93])     # array([8, 4, 3, 1]): the gain from each extra component
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ks, ____, marker="o", label="NMF")
ax[0].plot(ks, ____, marker="s", label="PCA")
ax[0].plot(ks, ____, ls="--", color="gray", label="random W (null)")
ax[0].axhline(____, ls=":", color="gray")
ax[0].set_xlabel("k"); ax[0].set_ylabel("test VAF (%)"); ax[0].legend()
ax[1].plot(ks, vaf_nmf_c, marker="o", label="NMF"); ax[1].plot(ks, vaf_pca_c, marker="s", label="PCA")
ax[1].set_xlabel("k"); ax[1].set_ylabel("centred test VAF (%)"); ax[1].legend(); plt.show()
K_CHOSEN = ____
```

</details>

✍️ **Q6.** Which rule did you use to choose k (flattening, 90%, or something else), and what result would make
you change it? Would two people looking at this curve pick the same k?

> *Your answer:*

---
## Task 7 · Choose k on held-out stages (leave-one-stage-out)
The curves above are fit on the start of each recording and scored on its end: the test frames come from the
**same tasks**. A stricter test of "how many synergies does this person use" is to hold out a **whole stage**:
fit on all the other stages, score on the held-out one, and repeat for every stage.

**Independent units.** The 30 recordings are only **15 independent blocks**: the left and right recordings of a
block were made **at the same time** (same block number in `filename_of_rec`). Holding out a stage must also hold
out anything recorded simultaneously with it, so the grouping (you write it in part 1 below; the check compares it with the shared
`protocol.leave_one_stage_out`) merges `OneHandedFreeStyle` with the idle-hand `unconstrained` recording made at the same time. That
gives **14 folds**. Every fold gives one number per k, and the spread over folds gives a **95% CI**
(your `bootstrap_ci`, which resamples folds, not frames).

**Rule for k:** the smallest k after which one more synergy adds less than **1 VAF point** on held-out stages
(mean over folds). Report the gap over random synergies at that k, with its CI.

**Keeping it fast:** 14 folds × 7 values of k = 98 NMF fits. Using every **10th** frame and **100** update rounds
keeps this to about a minute; neighbouring frames are near-copies (20 ms apart, smoothed at 10 Hz), so little is lost.

### Task 7, part 1 · The folds and the confidence interval, written by you
Every number in the later notebooks uses the same folds and the same kind of CI, so it is worth knowing exactly
what they do. You write four functions; the check compares them with the shared `protocol.py`.

**`block_of_rec`: which recordings were made at the same time?** The file name holds a recording number,
e.g. `...recording-13_left` and `...recording-13_right`. Same number = same block = recorded together. Renumber the
15 block numbers as 0 … 14.

**`stage_groups`: which stages must be held out together?** Most blocks have the same stage on both hands. But in
two blocks one hand did `OneHandedFreeStyle` while the idle hand's recording is labelled `unconstrained`. Holding
out `OneHandedFreeStyle` alone would leave its simultaneous `unconstrained` twin in the training set: a leak. So
two stages that share any block go in one group, and so on through chains of such links. This is a
**connected-components** problem: stages are the nodes, a shared block joins two nodes. Result: 14 groups.

**`leave_one_stage_out`: one fold per group.** Test = every frame whose recording is in the group;
train = every other frame.

**`bootstrap_ci`: how sure are we of a mean over 14 folds?** The folds are the independent units (frames 20 ms apart
are not), so resample **folds**: draw 14 fold values *with replacement*, take their mean, repeat 2000 times. The
spread of those 2000 means shows how much the mean would move with a different set of stages. The **percentile**
CI is the middle 95% of them: the 2.5% and 97.5% quantiles.

In [ ]:
import re


def block_of_rec(filename: np.ndarray) -> np.ndarray:
    """(R,) file names → (R,) int block id 0 … 14. Recordings made at the same time share an id.

    The block is the number after "recording-" in the name ("...recording-13_left..." → 13), then renumbered
    0, 1, 2, … in increasing order of that number.
    """
    # TODO
    raise NotImplementedError


def stage_groups(stage_of_rec: np.ndarray, filename: np.ndarray) -> list[tuple[str, np.ndarray]]:
    """Group recordings so that holding out a stage also holds out everything recorded at the same time.

    Two stages are in the same group if some block has one of them on one hand and the other on the other hand
    (and so on, through chains of such links).

    Returns
    -------
    list of (name, recs), sorted by name. name: the group's stage names, sorted, joined by " + ".
    recs: sorted int array of every recording id (0 … 29) in the group. 14 groups for this dataset.
    """
    # TODO
    raise NotImplementedError


def leave_one_stage_out(stage_of_rec: np.ndarray, rec: np.ndarray, filename: np.ndarray
                        ) -> list[tuple[str, np.ndarray, np.ndarray]]:
    """One fold per stage group, in the order of stage_groups: (name, train_mask, test_mask).

    rec : (N,) recording id of every frame. Masks are (N,) bool:
    test = frames whose recording is in the group; train = every other frame.
    """
    # TODO
    raise NotImplementedError


def bootstrap_ci(values: np.ndarray, n_boot: int = 2000, alpha: float = 0.05, seed: int = 0
                 ) -> tuple[float, float, float]:
    """Mean and (1 − alpha) percentile bootstrap CI, resampling the independent units (here: folds).

    Recipe (so your numbers match protocol.session_bootstrap_ci):
    1. Drop NaNs from values → v (S,). rng = np.random.default_rng(seed).
    2. Draw an (n_boot, S) table from v with replacement: rng.choice(v, size=(n_boot, len(v)), replace=True).
    3. The mean of each row → n_boot bootstrap means.
    4. Return (mean of v, alpha/2 quantile, 1 − alpha/2 quantile of the bootstrap means), as floats.
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on part 1? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `block_of_rec`: pull the number out of each file name, then map the distinct numbers to 0 … 14.
2. `stage_groups`: start with every stage in its own group. For each block, find the groups that contain any
   stage recorded in that block, and merge them into one. (One pass over the blocks is enough: a merged group
   keeps absorbing.) Then, for each group, collect its recordings and build its name.
3. `leave_one_stage_out`: for each group, `test` is "this frame's recording is one of the group's recordings".
4. `bootstrap_ci`: follow the four-step recipe in the docstring.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `re.search(r"recording-(\d+)_", s).group(1)` → the digits as a string; `int(...)` converts. Use `str(f)` on each
  name first (they are NumPy strings).
- `np.unique(nums, return_inverse=True)` returns `(distinct values, position of each value among them)`; the
  second is exactly a 0 … 14 renumbering.
- Sets: `a & b` (shared items, empty means no link), `set().union(*list_of_sets)` (merge many).
- `np.isin(rec, recs)` → (N,) bool: is each frame's recording in `recs`? `~mask` flips it.
- `np.quantile(x, 0.025)`; `v[~np.isnan(v)]` drops NaNs.

```python
np.unique([13, 2, 13, 7], return_inverse=True)   # (array([2, 7, 13]), array([2, 0, 2, 1]))
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def block_of_rec(filename):
    nums = np.array([int(re.search(r"recording-(\d+)_", str(f)).group(____)) for f in filename])
    _, bid = np.unique(nums, return_inverse=____)
    return bid

def stage_groups(stage_of_rec, filename):
    blk = block_of_rec(filename)
    stages = np.array([str(s) for s in stage_of_rec])
    groups = [{s} for s in np.unique(stages)]              # every stage alone
    for b in np.unique(blk):
        here = set(stages[blk == ____])                     # stages recorded in block b (usually 1, sometimes 2)
        touching = [g for g in groups if g & here]
        groups = [g for g in groups if not (g & here)] + [set().union(*____)]
    out = [(" + ".join(sorted(g)), np.where(np.isin(stages, list(g)))[0]) for g in groups]
    return sorted(out, key=lambda t: t[____])

def leave_one_stage_out(stage_of_rec, rec, filename):
    folds = []
    for name, recs in stage_groups(stage_of_rec, filename):
        te = np.isin(rec, ____)
        folds.append((name, ____, te))
    return folds

def bootstrap_ci(values, n_boot=2000, alpha=0.05, seed=0):
    v = np.asarray(values, float); v = v[~np.isnan(v)]
    rng = np.random.default_rng(seed)
    boots = rng.choice(v, size=(n_boot, len(v)), replace=True).mean(axis=____)
    return float(v.mean()), float(np.quantile(boots, alpha / 2)), float(np.quantile(boots, ____))
```

</details>

In [ ]:
# ✅ Check Task 7, part 1: your folds and CI against the shared protocol
show("block id of each of the 30 recordings", block_of_rec(filename_of_rec), protocol.block_of_rec(filename_of_rec))
assert np.array_equal(block_of_rec(filename_of_rec), protocol.block_of_rec(filename_of_rec)), \
    "block ids differ: the number after 'recording-', renumbered 0 … 14 in increasing order"
mine, theirs = stage_groups(stage_of_rec, filename_of_rec), protocol.stage_groups(stage_of_rec, filename_of_rec)
show("number of stage groups", len(mine), len(theirs))
show("the merged group", next((f"{n}: recordings {np.asarray(r).tolist()}" for n, r in mine if "+" in n), "none found"),
     next(f"{n}: recordings {np.asarray(r).tolist()}" for n, r in theirs if "+" in n))
assert len(mine) == 14, f"{len(mine)} groups, expected 14: OneHandedFreeStyle and unconstrained share blocks, so they merge"
for (n1, r1), (n2, r2) in zip(mine, theirs):
    assert n1 == n2, f"group names differ: yours {n1!r}, expected {n2!r} (sorted stage names joined by ' + ', list sorted by name)"
    assert np.array_equal(np.sort(r1), r2), f"group {n2!r}: recordings {np.sort(r1)}, expected {r2}"
my_folds = leave_one_stage_out(stage_of_rec, rec, filename_of_rec)
ref_folds = list(protocol.leave_one_stage_out(stage_of_rec, rec, filename=filename_of_rec))
show("test frames in the first 3 folds", np.array([te.sum() for _, _, te in my_folds[:3]]),
     np.array([te.sum() for _, _, te in ref_folds[:3]]))
assert len(my_folds) == len(ref_folds) == 14
for (n1, tr1, te1), (n2, tr2, te2) in zip(my_folds, ref_folds):
    assert not (tr1 & te1).any(), f"fold {n1!r}: a frame is in both train and test"
    assert np.array_equal(te1, te2) and np.array_equal(tr1, tr2), f"fold {n2!r}: masks differ from the protocol"
v_toy = np.array([1.0, 2.0, 4.0, np.nan, 3.0, 2.5])
show("bootstrap_ci on a toy example: (mean, low, high)", np.array(bootstrap_ci(v_toy)), np.array(protocol.session_bootstrap_ci(v_toy)))
assert np.allclose(bootstrap_ci(v_toy), protocol.session_bootstrap_ci(v_toy)), \
    f"bootstrap_ci {np.round(bootstrap_ci(v_toy), 4)}, expected {np.round(protocol.session_bootstrap_ci(v_toy), 4)}: follow the recipe"
print(f"✅ Your 14 leave-one-stage-out folds and your bootstrap CI match the shared protocol")

In [ ]:
def heldout_vaf_table(X: np.ndarray, folds: list[tuple[np.ndarray, np.ndarray]], ks: list[int],
                      step: int = 10, n_iter: int = 100, seed: int = 0) -> tuple[np.ndarray, np.ndarray]:
    """Held-out VAF for every fold and k, for NMF and for random_W of the same size.

    Parameters
    ----------
    X     : (N, 16) EMG
    folds : list of (train_mask, test_mask), each (N,) bool
    ks    : values of k to try
    step, n_iter : use X[mask][::step] and n_iter NMF rounds (both for nmf_fit and nmf_transform)

    Returns
    -------
    real, rand : each (n_folds, len(ks)) float, VAF (%) on the test frames of each fold
    """
    # TODO
    raise NotImplementedError


ks_loso = list(range(1, 8))
folds = [(tr, te) for _, tr, te in leave_one_stage_out(stage_of_rec, rec, filename_of_rec)]   # your folds
real_loso, rand_loso = None, None      # TODO: heldout_vaf_table(emg, folds, ks_loso, seed=SEED)
K_LOSO: int | None = None              # TODO: smallest k whose next synergy adds < 1 point (mean over folds)
gap_ci = None                          # TODO: bootstrap_ci(real − rand at K_LOSO, per fold)

#### 💡 Stuck on Task 7? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Loop over folds and over k. In each fold, take every `step`-th training and test frame.
2. Fit NMF on the training frames, then rebuild the test frames with `nmf_transform` and score with `vaf`.
3. Do the same rebuild with `random_W(k, 16, seed)` instead of the fitted `W`.
4. `K_LOSO`: `np.diff(real_loso.mean(axis=0))` gives the gain of each extra synergy; find the first gain below 1.
5. `gap_ci`: the per-fold difference `real_loso[:, j] - rand_loso[:, j]` at that k goes into the bootstrap.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.zeros((len(folds), len(ks)))` makes the empty tables; fill `real[f, j]`.
- `for f, (tr, te) in enumerate(folds):` gives the fold number and its two masks.
- `bootstrap_ci(values)` (yours, from part 1) → `(mean, low, high)`.

```python
gains = np.diff([75, 80, 87, 90, 92.5, 93.8, 94.0])   # [5, 7, 3, 2.5, 1.3, 0.2]
next(i for i, g in enumerate(gains) if g < 1)          # 5 → k = ks[5] = 6
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def heldout_vaf_table(X, folds, ks, step=10, n_iter=100, seed=0):
    real = np.zeros((len(folds), len(ks))); rand = np.zeros_like(real)
    for f, (tr, te) in enumerate(folds):
        X_tr, X_te = X[tr][::step], X[te][::step]
        for j, k in enumerate(ks):
            _, W, _ = nmf_fit(____, k, n_iter=n_iter, seed=seed)
            real[f, j] = vaf(X_te, nmf_transform(____, W, n_iter=n_iter, seed=seed) @ W)
            Wr = random_W(k, X.shape[1], seed)
            rand[f, j] = vaf(X_te, nmf_transform(X_te, ____, n_iter=n_iter, seed=seed) @ Wr)
    return real, rand

real_loso, rand_loso = heldout_vaf_table(emg, folds, ks_loso, seed=SEED)
gains = np.diff(real_loso.mean(axis=0))
K_LOSO = ks_loso[next((i for i, g in enumerate(gains) if g < ____), len(gains))]
j = ks_loso.index(K_LOSO)
gap_ci = bootstrap_ci(real_loso[:, j] - rand_loso[:, ____])
```

</details>

In [ ]:
# ✅ Check Task 7 (about 10 s)
assert len(folds) == 14, "14 folds: OneHandedFreeStyle and unconstrained were recorded together and are held out together"
assert real_loso.shape == rand_loso.shape == (14, len(ks_loso))
show("held-out VAF (%) for k = 1 … 7, mean over the 14 folds: NMF", real_loso.mean(axis=0))
show("held-out VAF (%) for k = 1 … 7, mean over the 14 folds: random W", rand_loso.mean(axis=0))
r_real, r_rand = ref.heldout_vaf_table(emg, folds[:1], [4], step=10, n_iter=100, seed=SEED)
show("fold 0, k = 4: (NMF, random W)", np.array([real_loso[0, 3], rand_loso[0, 3]]), np.array([r_real[0, 0], r_rand[0, 0]]))
show("K_LOSO", K_LOSO, ref.choose_k(real_loso, ks_loso))
show("gap_ci: (mean, low, high) of NMF − random at K_LOSO", "not set yet" if gap_ci is None else np.array(gap_ci))
assert abs(real_loso[0, 3] - r_real[0, 0]) < 0.5 and abs(rand_loso[0, 3] - r_rand[0, 0]) < 0.5, \
    f"fold 0, k = 4: yours {real_loso[0, 3]:.2f} / {rand_loso[0, 3]:.2f}, reference {r_real[0, 0]:.2f} / {r_rand[0, 0]:.2f}"
assert K_LOSO == ref.choose_k(real_loso, ks_loso), f"K_LOSO = {K_LOSO}, expected {ref.choose_k(real_loso, ks_loso)}"
m, lo, hi = gap_ci
print(f"✅ Leave-one-stage-out chooses k = {K_LOSO}; NMF beats random synergies there by {m:.1f} points "
      f"(95% CI {lo:.1f} to {hi:.1f}, over 14 held-out stage groups)")

### 📊 Held-out stages: real vs random synergies, and the gain from each extra synergy
Left: mean held-out VAF ± spread over folds, for NMF and for random synergies. Right: how much each extra synergy
adds, with the 1-point rule. **What to look for:** a wide, steady gap between the two curves (the synergies are
real structure), and where the right-hand bars drop below the line (your k).

In [ ]:
# TODO: a 12 × 4 figure with 2 panels.
# TODO: left: mean held-out VAF over folds against ks_loso, with the std over folds as error bars, for real_loso
#       ("NMF") and rand_loso ("random W (null)", grey). Axis labels and a legend.
# TODO: right: the gain from each extra synergy, np.diff(real_loso, axis=1), as bars at ks_loso[1:] (mean ± std over
#       folds), with a dashed line at 1 (the rule for k). Label x "k (gain from k − 1 to k)" and y "VAF points".
#       (Tool: ax.errorbar(x, y, yerr=..., marker="o", capsize=3); ax.bar(x, y, yerr=..., capsize=3).)

✍️ **Q7.** Does leave-one-stage-out pick the same k as your Task 6 choice? If not, which would you trust for the decoder, and
why? Why does the protocol hold out `OneHandedFreeStyle` and `unconstrained` together, and what would go wrong if
it didn't?

> *Your answer:*

---
## Task 8 · Look at the synergies
For the chosen k:
1. A bar chart per synergy of its 16 electrode weights (`nmf_W[K_CHOSEN]`, rows already length 1).
2. For one recording: the activations `C` over time, with 3 joint angles underneath on the same time axis.
   When does each synergy switch on? Does it line up with a particular finger moving?

In [ ]:
W = nmf_W[K_CHOSEN]                      # (k, 16): row i = synergy i
r = 0                                    # the recording to plot
in_r = rec == r                          # (N,) bool mask for that recording
t_r = np.arange(in_r.sum()) / 50         # time axis in seconds (50 frames/s)

# TODO: C_r = nmf_transform(emg[in_r], W, seed=SEED)    → (T_r, k) activations of recording r
# TODO: a row of bar charts, one per synergy (its 16 weights)
# TODO: a 2-row figure sharing the x axis: C_r over time (top); pose[in_r] for 3 joints of your choice (bottom)

#### 💡 Stuck on Task 8? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Compute the activations of the recording with your `nmf_transform` and the chosen `W`.
2. Figure 1: `plt.subplots(1, K_CHOSEN)` and a `bar` per synergy.
3. Figure 2: `plt.subplots(2, 1, sharex=True)`: one line per synergy on top, three joint angles underneath.
   Masked pose frames (`pose_valid` False) sit at 0°; you can blank them with `np.where(pose_valid[in_r], …, np.nan)`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `plt.subplots(2, 1, sharex=True)` → two axes stacked; zooming one zooms both.
- `ax.plot(t, Y)` with a 2-D `Y` (T, n) draws one line per column.

```python
y = np.array([1.0, 2.0, 3.0]); ok = np.array([True, False, True])
np.where(ok, y, np.nan)    # array([ 1., nan,  3.]): NaN points are not drawn
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
C_r = nmf_transform(emg[in_r], ____, seed=SEED)
fig, ax = plt.subplots(1, K_CHOSEN, figsize=(14, 2.5), sharey=True)
for i in range(K_CHOSEN):
    ax[i].bar(range(16), ____[i]); ax[i].set_title(f"synergy {i + 1}")
plt.show()

joints = [5, 9, 13]                         # INDEX_, MIDDLE_, RING_MCP_FE
P_r = np.where(pose_valid[in_r][:, None], pose[in_r][:, joints], np.nan)
fig, ax = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
ax[0].plot(t_r, ____); ax[0].set_ylabel("activation")
ax[1].plot(t_r, ____); ax[1].set_ylabel("angle (°)"); ax[1].set_xlabel("time (s)")
plt.show()
```

</details>

✍️ **Q8.** Does any synergy switch on with a particular finger movement? Does a synergy sit on neighbouring
electrodes (one region of the forearm)? What else, apart from a shared neural command, could make
neighbouring electrodes rise together?

> *Your answer:*

---
## Task 9 · Eigenpostures: how many dimensions does the pose need?
The proposal's **pose bottleneck**: the decoder predicts a few PCA scores, and PCA turns them back into 20 angles.
Fingers share tendons, so joint angles move together and a few components should rebuild them well.

Your `pca_fit` works on any (T, D) data, so use it again on the pose: fit on **valid** training poses only
(`pose_valid`), rebuild the **test** poses with k = 1 … 20 components, and measure the error in degrees.

**Choosing `K_POSE` is a judgement call; agree it as a team.** The bottleneck's error adds to the decoder's,
so compare it with the error we want to beat (the best published model is off by 15.63°) and with a decoder
that always guesses the average pose. The proposal's starting rule was **under 2°**: check whether that is
reachable with *few* components, and if not, say what that means for the eigenposture idea (Santello et al.
reported that 2 components explain most of the *variance* of static grasps: is mean error in degrees on
free movement the same question?).

In [ ]:
P_train = pose[train & pose_valid]   # (T_train_valid, 20) degrees
P_test  = pose[test & pose_valid]    # (T_test_valid, 20)


def pose_error_deg(Y: np.ndarray, Y_hat: np.ndarray) -> float:
    """Mean absolute joint-angle error in degrees.

    Parameters
    ----------
    Y     : (T, 20) float, true joint angles in degrees
    Y_hat : (T, 20) float, rebuilt joint angles in degrees

    Returns
    -------
    float, averaged over all joints and frames. Check: pose_error_deg(Y, Y) → 0.0
    """
    # TODO
    raise NotImplementedError


ks_pose = list(range(1, 21))
err_pose: list[float] = []                              # length 20 when done: test error (°) for k = 1 … 20
pose_fits: dict[int, tuple[np.ndarray, np.ndarray]] = {}  # k → (mean, comps)

for k in ks_pose:
    # TODO: pca_fit on P_train, store it in pose_fits[k]
    # TODO: rebuild P_test, append pose_error_deg(P_test, rebuilt) to err_pose
    ...

# TODO: plot err_pose against ks_pose, with a dashed line at your cut-off
K_POSE: int | None = None            # TODO: your choice, and the reason in the Result cell

#### 💡 Stuck on Task 9? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `pose_error_deg`: the absolute difference at every entry, then the mean of all of them.
2. The loop is the PCA half of Task 4, on `P_train` / `P_test`, with `pose_error_deg` instead of `vaf`.
3. Plot like Task 6, with the dashed line at your cut-off, and pick the first k below it.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.abs(a)` → absolute value of every entry; `np.mean(a)` with no `axis=` → one number.
- To find the first k under a cut-off: `next(k for k, e in zip(ks_pose, err_pose) if e < 2.0)`.

```python
np.mean(np.abs(np.array([[1.0, -3.0]]) - 0))   # 2.0
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def pose_error_deg(Y, Y_hat):
    return float(np.mean(np.abs(____ - ____)))

for k in ks_pose:
    mean, comps = pca_fit(____, k)
    pose_fits[k] = (mean, comps)
    P_hat = pca_inverse(pca_transform(____, mean, comps), mean, comps)
    err_pose.append(pose_error_deg(____, ____))
```

</details>

In [ ]:
# ✅ Check Task 9
Y_toy = np.random.default_rng(1).normal(size=(30, 20))
show("pose_error_deg(Y, Y): should be 0", pose_error_deg(Y_toy, Y_toy), 0.0)
show("pose_error_deg(Y, Y + 2): should be 2", pose_error_deg(Y_toy, Y_toy + 2), 2.0)
assert pose_error_deg(Y_toy, Y_toy) == 0.0
assert abs(pose_error_deg(Y_toy, Y_toy + 2) - 2.0) < 1e-9, "an error of 2° everywhere should give 2.0"
assert len(err_pose) == 20, "one error per k = 1 … 20"
r_err = [ref.pose_error_deg(P_test, ref.pca_inverse(ref.pca_transform(P_test, *ref.pca_fit(P_train, k)),
                                                    *ref.pca_fit(P_train, k))) for k in ks_pose]
show("test pose error (°) for k = 1 … 20", np.array(err_pose), np.array(r_err))
assert np.allclose(err_pose, r_err, atol=1e-6), "errors differ from the reference: fit on P_train, score on P_test"
print("✅ Pose errors (°): " + ", ".join(f"k={k}: {e:.1f}" for k, e in zip(ks_pose, err_pose)))

✍️ **Q9.** The pose needs many more PCA components than the EMG needs synergies to reach a small error. Give
two possible reasons (think about what 16 electrodes on the skin can resolve, how independent the fingers are,
and what is in the tracking data). Santello et al. found 2 components explain most of the variance of *static
grasps*. Is mean error in degrees on *free movement* the same question?

> *Your answer:*

---
## ⚠️ Methodology: what this notebook can and can't tell you
Read this before writing the Result. Each row is a reason the numbers could look better (or different) than the truth, and what would test it.

| Issue | Why it matters | What would test it |
|---|---|---|
| **15 independent blocks, not 30 recordings** | The left- and right-hand recordings of a block were made at the same time, and blocks 14–15 pair OneHandedFreeStyle with the idle hand's 'unconstrained' recording. Treating 30 recordings (or frames) as independent overstates certainty | Every CI here resamples held-out stage groups (14 folds) via `protocol`, with `filename=` |
| **One person** | All 15 blocks are the same participant | Nothing here says how many synergies *other* people need; that needs the full dataset (stage 7) |
| **Uncentred VAF is generous** | The training mean alone scores well above 0 (Task 5) | Report centred VAF and the gap over random synergies, not only VAF |
| **`C` is refit on the test frames** | Random synergies already score high, and VAF can only rise with k | The random-W null (Task 5, Task 7); for an over-fitting-aware k, **bi-cross-validation** (hold out channels *and* frames, Owen & Perry 2009) |
| **Time split vs held-out stages** | The time split tests later frames of the same tasks | Task 7 holds out whole stage groups; prefer its k |
| **Left and right hands pooled** | The band is mirrored on the other wrist, so electrode *j* sits over a different muscle on each hand | Fit each hand separately (notebook 03) and compare k |
| **Speed shortcuts in Task 7** | Every 10th frame and 100 NMF rounds make each fold's number slightly noisier | Rerun a borderline k at full size |
| **Noise floor** | Rectified background noise is a positive level that NMF may spend a synergy on | Subtract each channel's resting level (e.g. 5th percentile) and refit |
| **Loud channels dominate VAF** | One scale per recording, so loud electrodes carry more squared error | Repeat with per-channel scaling; does k change? |
| **Pose error includes tracking noise** | Small eigenpostures may fit camera/IK jitter | Low-pass the pose at ~10 Hz before PCA, or report variance explained as well as degrees |

---
## Result · gate for stage 1
**Question:** Is the EMG low-dimensional beyond chance, and how many synergies (k) and eigenpostures (K_POSE)?

| Measure | Value (mean, 95% CI over sessions) | Baseline / null | Pass? |
|---|---|---|---|
| k chosen on the time split (Task 6) | | | |
| k chosen by leave-one-stage-out (Task 7) | | | |
| VAF / centred VAF at that k (NMF; PCA) | | | |
| Gap over random synergies at that k (Task 7 CI) | | | |
| Chosen K_POSE and pose error at K_POSE (°) | | | |

**Gate:** the gap over random synergies at the leave-one-stage-out k has a 95% CI above 0 (and is large, e.g. > 5 points).

- **Pass →** stage 2 (notebook 03): are the synergies stable and not crosstalk? Use the leave-one-stage-out k.
- **Fail →** the EMG is not lower-dimensional than chance co-activation: stop calling the components synergies; stages 4–5 can still ask what the EMG encodes.
- **Was your prediction right? Why or why not?**